In [1]:
# CIR模型分析

In [1]:
import os
import sys
import glob
import subprocess
import pandas as pd

# =========================================================
# 0) 設定
# =========================================================
seeds = [1, 2, 3, 4, 5]

exp_root = "./experiments_none_only"
subset_ids_path = "/home/ester/valid_description/no_missing_none_ids.txt"
run_tag = "none_subset"

text_variants = {
    "outfitUrlTitle": "encoded_outfitUrlTitle_en_fashionClip.pkl",
    "NewoutfitUrlTitle": "encoded_NewoutfitUrlTitle_en_fashionClip.pkl",
    "no_weather": "encoded_no_weather_outfitUrlTitle_en_fashionClip.pkl",
    "no_occasion": "encoded_no_occasion_outfitUrlTitle_en_fashionClip.pkl",
    "no_style": "encoded_no_style_outfitUrlTitle_en_fashionClip.pkl",
}

DETAIL_DIR = os.path.join(exp_root, "detail_cir")
os.makedirs(DETAIL_DIR, exist_ok=True)

# =========================================================
# 1) 前置檢查
# =========================================================
if not os.path.exists(subset_ids_path):
    raise FileNotFoundError(f"找不到 subset ids 檔案：{subset_ids_path}")

print("Python:", sys.executable)
print("exp_root:", exp_root)
print("subset_ids_path:", subset_ids_path)
print("detail_dir:", DETAIL_DIR)

# =========================================================
# 2) 執行函式
# =========================================================
def expected_detail_path(seed, variant):
    subset_tag = "subset" if subset_ids_path else "all"
    fname = f"detail_cir_{variant}_{subset_tag}_{run_tag}_seed{seed}.csv"
    return os.path.join(DETAIL_DIR, fname)

def run_cir_detail(seed, variant, outfit_feat_name):
    detail_file = expected_detail_path(seed, variant)

    if os.path.exists(detail_file):
        print(f"[SKIP] seed={seed}, variant={variant}: 已存在 -> {detail_file}")
        return detail_file

    cmd = [
        sys.executable, "-u", "evaluate_cir.py",
        f"--seed={seed}",
        f"--exp_root={exp_root}",
        f"--variant={variant}",
        f"--outfit_feat_name={outfit_feat_name}",
        f"--subset_ids_path={subset_ids_path}",
        f"--run_tag={run_tag}",
        "--save_detail",
        f"--detail_file={detail_file}",
    ]

    print(f"[RUN ] seed={seed}, variant={variant}")
    print("      CMD =", " ".join(cmd))

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    subprocess.run(cmd, check=True, env=env)
    return detail_file

# =========================================================
# 3) 主流程：產生 25 個 detail csv
# =========================================================
generated_files = []

for seed in seeds:
    print("\n========================")
    print(f"===== seed = {seed} =====")
    print("========================\n")

    for variant, outfit_feat_name in text_variants.items():
        fp = run_cir_detail(seed, variant, outfit_feat_name)
        generated_files.append({
            "seed": seed,
            "variant": variant,
            "detail_file": fp,
            "exists": os.path.exists(fp),
        })

# =========================================================
# 4) 檢查輸出
# =========================================================
manifest = pd.DataFrame(generated_files)

# 檢查欄位是否存在（抽查已生成檔案）
required_cols = [
    "run_tag", "seed", "set_id", "target_item_id", "target_item_fg",
    "rank", "hit@1", "hit@3", "hit@5", "hit@10", "hit@30", "hit@50",
    "top10_ids"
]

check_rows = []
for fp in manifest["detail_file"]:
    if os.path.exists(fp):
        try:
            df = pd.read_csv(fp, nrows=3)
            cols = set(df.columns)
            ok = all(c in cols for c in required_cols)
            check_rows.append({
                "detail_file": fp,
                "n_columns": len(cols),
                "schema_ok": ok,
                "missing_cols": ",".join([c for c in required_cols if c not in cols]),
            })
        except Exception as e:
            check_rows.append({
                "detail_file": fp,
                "n_columns": -1,
                "schema_ok": False,
                "missing_cols": f"READ_ERROR: {e}",
            })

check_df = pd.DataFrame(check_rows)

print("\n=== detail manifest ===")
display(manifest)

print("\n=== schema check ===")
display(check_df)

# =========================================================
# 5) 只保存真正必要的追蹤檔
# =========================================================
manifest.to_csv(os.path.join(DETAIL_DIR, "manifest.csv"), index=False)
check_df.to_csv(os.path.join(DETAIL_DIR, "schema_check.csv"), index=False)

print("\n✅ 全部 CIR detail csv 產生完成")
print(f"manifest -> {os.path.join(DETAIL_DIR, 'manifest.csv')}")
print(f"schema   -> {os.path.join(DETAIL_DIR, 'schema_check.csv')}")

Python: /home/ester/miniforge3/envs/minenv/bin/python
exp_root: ./experiments_none_only
subset_ids_path: /home/ester/valid_description/no_missing_none_ids.txt
detail_dir: ./experiments_none_only/detail_cir

===== seed = 1 =====

[RUN ] seed=1, variant=outfitUrlTitle
      CMD = /home/ester/miniforge3/envs/minenv/bin/python -u evaluate_cir.py --seed=1 --exp_root=./experiments_none_only --variant=outfitUrlTitle --outfit_feat_name=encoded_outfitUrlTitle_en_fashionClip.pkl --subset_ids_path=/home/ester/valid_description/no_missing_none_ids.txt --run_tag=none_subset --save_detail --detail_file=./experiments_none_only/detail_cir/detail_cir_outfitUrlTitle_subset_none_subset_seed1.csv


2026-03-17 18:24:16.021516: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-17 18:24:16.236999: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773743056.318414  127359 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773743056.341865  127359 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773743056.531661  127359 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

[CIR EVAL] seed=1, variant=outfitUrlTitle, subset_tag=subset, exp_dir=./experiments_none_only/cir_outfitUrlTitle_subset_seed1
always_save_checkpoint                  True                                    
batch_size                              50                                      
category_feat_path                      './encoded_category_distiluse-base-multilingual-cased-v2.pkl'
compile_model                           False                                   
datadir                                 'E:\\Datasets\\UIUC-polyvore'           
decay_lr                                True                                    
device                                  device(type='cuda', index=0)            
device_type                             'cuda'                                  
dropout                                 0.1                                     
dtype                                   'float16'                               
epochs                                  100

compute recall@topk:   0%|          | 0/9930 [00:00<?, ?it/s]

[8/151] fine-grained categories are selected.


...

Recall@top1: 0.9615%
Recall@top3: 2.7098%
Recall@top5: 4.2541%
Recall@top10: 7.2844%
Recall@top30: 15.4720%
Recall@top50: 21.0664%
[CIR EVAL] 結果已寫入 ./experiments_none_only/results_cir.csv
[CIR none_subset] detail 已寫入 ./experiments_none_only/detail_cir/detail_cir_no_style_subset_none_subset_seed5.csv

=== detail manifest ===


,seed,variant,detail_file,exists
0,1,outfitUrlTitle,./experiments_none_only/detail_cir/detail_cir_...,True
1,1,NewoutfitUrlTitle,./experiments_none_only/detail_cir/detail_cir_...,True
2,1,no_weather,./experiments_none_only/detail_cir/detail_cir_...,True
3,1,no_occasion,./experiments_none_only/detail_cir/detail_cir_...,True
4,1,no_style,./experiments_none_only/detail_cir/detail_cir_...,True
5,2,outfitUrlTitle,./experiments_none_only/detail_cir/detail_cir_...,True
6,2,NewoutfitUrlTitle,./experiments_none_only/detail_cir/detail_cir_...,True
7,2,no_weather,./experiments_none_only/detail_cir/detail_cir_...,True
8,2,no_occasion,./experiments_none_only/detail_cir/detail_cir_...,True
9,2,no_style,./experiments_none_only/detail_cir/detail_cir_...,True



=== schema check ===


,detail_file,n_columns,schema_ok,missing_cols
0,./experiments_none_only/detail_cir/detail_cir_...,13,True,
1,./experiments_none_only/detail_cir/detail_cir_...,13,True,
2,./experiments_none_only/detail_cir/detail_cir_...,13,True,
3,./experiments_none_only/detail_cir/detail_cir_...,13,True,
4,./experiments_none_only/detail_cir/detail_cir_...,13,True,
5,./experiments_none_only/detail_cir/detail_cir_...,13,True,
6,./experiments_none_only/detail_cir/detail_cir_...,13,True,
7,./experiments_none_only/detail_cir/detail_cir_...,13,True,
8,./experiments_none_only/detail_cir/detail_cir_...,13,True,
9,./experiments_none_only/detail_cir/detail_cir_...,13,True,



✅ 全部 CIR detail csv 產生完成
manifest -> ./experiments_none_only/detail_cir/manifest.csv
schema   -> ./experiments_none_only/detail_cir/schema_check.csv


In [33]:
import os
import re
import json
import glob
import ast
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from IPython.display import display

# =========================================================
# 0) CONFIG
# =========================================================
EXP_ROOT = "./experiments_none_only"
DETAIL_DIR = os.path.join(EXP_ROOT, "detail_cir")
if not os.path.exists(DETAIL_DIR):
    DETAIL_DIR = EXP_ROOT

DETAIL_RUN_TAG    = "none_subset"
DETAIL_SUBSET_TAG = "subset"

# 新描述 / fragments / ablation 來源
FRAG_JSONL = "/home/ester/valid_description/wos_split_results_v5_merged_retry_round3.jsonl"

# 原始描述來源
ORIGINAL_TITLE_JSON = "/home/ester/valid_description/polyvore_data/polyvore_outfits/polyvore_outfit_titles.json"

# Category mapping CSV
CATEGORY_CSV = "/home/ester/valid_description/polyvore_data/polyvore_outfits/categories.csv"

# 圖片路徑
IMAGE_ROOTS = [
    "/home/ester/valid_description/polyvore_data/polyvore_outfits/images",
]

SAVE_TABLES = False
SAVE_FIGS   = False      # True → 存 PDF/PNG

OUTPUT_DIR = "./qualitative_outputs_final"
TABLE_DIR  = os.path.join(OUTPUT_DIR, "tables")
FIG_DIR    = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TABLE_DIR,  exist_ok=True)
os.makedirs(FIG_DIR,    exist_ok=True)

VARIANTS = {
    "original"   : "outfitUrlTitle",
    "full"       : "NewoutfitUrlTitle",
    "no_weather" : "no_weather",
    "no_occasion": "no_occasion",
    "no_style"   : "no_style",
}

TOPK = 5

plt.rcParams.update({
    "figure.dpi"         : 160,
    "font.size"          : 10,
    "axes.spines.top"    : False,
    "axes.spines.right"  : False,
    "font.family"        : "DejaVu Sans",
})


# =========================================================
# 0b) Category mapping
# =========================================================
def load_category_map(csv_path):
    """
    讀取 categories.csv。
    支援格式：col0=id, col1=name, col2=group（無 header）
    回傳 dict: {str(id): "name (group)"}  例如 "dress (all-body)"
    若無 group 欄則僅回傳 name。
    同一 id 可能對應多個 name（取第一筆）。
    """
    if not os.path.exists(csv_path):
        print(f"[WARNING] 找不到 categories.csv：{csv_path}")
        print("         → category 欄位將顯示原始數字 ID")
        return {}

    # 先嘗試無 header 讀取，若第一欄為數字即為無 header 格式
    df_raw = pd.read_csv(csv_path, header=None)
    first_val = str(df_raw.iloc[0, 0]).strip()
    if first_val.isdigit():
        df = df_raw.copy()
    else:
        # 有 header：自動偵測欄位
        df = pd.read_csv(csv_path)
        id_candidates   = ["id", "category_id", "fg"]
        name_candidates = ["name", "category_name", "category"]
        grp_candidates  = ["group", "supercategory", "type", "parent"]
        id_col  = next((c for c in id_candidates   if c in df.columns), df.columns[0])
        nm_col  = next((c for c in name_candidates if c in df.columns), df.columns[1])
        grp_col = next((c for c in grp_candidates  if c in df.columns), None)
        df = df.rename(columns={id_col: 0, nm_col: 1})
        if grp_col:
            df = df.rename(columns={grp_col: 2})

    has_group = df.shape[1] >= 3

    result = {}
    for _, row in df.iterrows():
        cid = str(row[0]).strip()
        if cid in result:          # 同 id 取第一筆
            continue
        name = str(row[1]).strip() if pd.notna(row[1]) else cid
        if has_group and pd.notna(row[2]):
            grp   = str(row[2]).strip()
            label = f"{name} ({grp})"
        else:
            label = name
        result[cid] = label

    sample = dict(list(result.items())[:4])
    print(f"[categories.csv] 載入 {len(result)} 個類別，範例：{sample}")
    return result


CATEGORY_NAME_MAP = load_category_map(CATEGORY_CSV)


def map_category(x):
    """數字 → 類別名稱；找不到就直接回傳原始字串。"""
    return CATEGORY_NAME_MAP.get(str(x), str(x))


# =========================================================
# 1) 讀取 detail csv
# =========================================================
def load_one_variant_detail(variant_name):
    pattern = os.path.join(
        DETAIL_DIR,
        f"detail_cir_{variant_name}_{DETAIL_SUBSET_TAG}_{DETAIL_RUN_TAG}_seed*.csv"
    )
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"找不到 detail 檔案：{pattern}")

    dfs = []
    for fp in files:
        df = pd.read_csv(fp)
        df["variant"]     = variant_name
        df["source_file"] = os.path.basename(fp)
        dfs.append(df)

    return pd.concat(dfs, axis=0, ignore_index=True)


detail_original   = load_one_variant_detail(VARIANTS["original"])
detail_full       = load_one_variant_detail(VARIANTS["full"])
detail_no_weather = load_one_variant_detail(VARIANTS["no_weather"])
detail_no_occasion= load_one_variant_detail(VARIANTS["no_occasion"])
detail_no_style   = load_one_variant_detail(VARIANTS["no_style"])

print("Loaded rows:")
for name, df in [("original", detail_original), ("full", detail_full),
                 ("no_weather", detail_no_weather), ("no_occasion", detail_no_occasion),
                 ("no_style", detail_no_style)]:
    print(f"  {name:12s}: {len(df)}")


# =========================================================
# 2) Schema 正規化
# =========================================================
def normalize_detail_df(df):
    out = df.copy()
    required = ["seed", "set_id", "target_item_id", "target_item_fg",
                "rank", "hit@10", "top10_ids"]
    for c in required:
        if c not in out.columns:
            raise KeyError(f"detail csv 缺少必要欄位：{c}")

    out["seed"]           = out["seed"].astype(int)
    out["set_id"]         = out["set_id"].astype(str)
    out["target_item_id"] = out["target_item_id"].astype(str)
    out["target_item_fg"] = out["target_item_fg"].astype(str)
    out["rank"]           = pd.to_numeric(out["rank"],    errors="coerce")
    out["hit@10"]         = pd.to_numeric(out["hit@10"],  errors="coerce")

    def parse_top10(x):
        if pd.isna(x):
            return []
        if isinstance(x, list):
            return [str(i) for i in x]
        try:
            return [str(i) for i in json.loads(x)]
        except Exception:
            try:
                v = ast.literal_eval(x)
                return [str(i) for i in v] if isinstance(v, list) else []
            except Exception:
                return []

    out["top10_ids_list"] = out["top10_ids"].apply(parse_top10)
    out["query_key"] = (
        out["seed"].astype(str) + "||" +
        out["set_id"]           + "||" +
        out["target_item_id"]
    )
    return out


detail_original    = normalize_detail_df(detail_original)
detail_full        = normalize_detail_df(detail_full)
detail_no_weather  = normalize_detail_df(detail_no_weather)
detail_no_occasion = normalize_detail_df(detail_no_occasion)
detail_no_style    = normalize_detail_df(detail_no_style)


# =========================================================
# 3) 載入 metadata
# =========================================================
def normalize_text(x):
    if x is None:
        return ""
    if isinstance(x, str):
        return x.strip()
    if isinstance(x, list):
        return " | ".join(str(i).strip() for i in x if str(i).strip())
    return str(x).strip()


def load_original_titles_json(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"找不到原始描述檔案：{path}")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []
    for sid, value in data.items():
        original_text = original_title_only = original_url_part = ""
        if isinstance(value, str):
            original_text = value.strip()
        elif isinstance(value, dict):
            title_part = normalize_text(value.get("title", ""))
            url_part   = normalize_text(value.get("url_name", value.get("url", "")))
            original_text      = " ".join(p for p in [url_part, title_part] if p).strip()
            original_title_only = title_part
            original_url_part   = url_part
        else:
            original_text = normalize_text(value)

        rows.append({
            "set_id"            : str(sid),
            "original_text"     : original_text,
            "original_title_only": original_title_only,
            "original_url_part" : original_url_part,
        })
    return pd.DataFrame(rows)


def load_fragments_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"找不到 fragments 檔案：{path}")
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if obj.get("valid") is not True:
                continue

            frags          = obj.get("fragments", {})
            title_ablation = obj.get("title_ablation", {})
            weather_terms  = frags.get("weather",  [])
            occasion_terms = frags.get("occasion", [])
            style_terms    = frags.get("style",    [])

            rows.append({
                "set_id"               : str(obj.get("id", "")),
                "title_full_text"      : normalize_text(obj.get("title", "")),
                "weather_terms"        : weather_terms  if isinstance(weather_terms,  list) else [],
                "occasion_terms"       : occasion_terms if isinstance(occasion_terms, list) else [],
                "style_terms"          : style_terms    if isinstance(style_terms,    list) else [],
                "weather_text"         : normalize_text(weather_terms),
                "occasion_text"        : normalize_text(occasion_terms),
                "style_text"           : normalize_text(style_terms),
                "title_no_weather_text" : normalize_text(title_ablation.get("no_weather",  "")),
                "title_no_occasion_text": normalize_text(title_ablation.get("no_occasion", "")),
                "title_no_style_text"   : normalize_text(title_ablation.get("no_style",    "")),
            })
    return pd.DataFrame(rows)


orig_title_df = load_original_titles_json(ORIGINAL_TITLE_JSON)
frag_df       = load_fragments_jsonl(FRAG_JSONL)

print(f"Loaded original titles : {len(orig_title_df)}")
display(orig_title_df.head(3))
print(f"Loaded fragments       : {len(frag_df)}")
display(frag_df.head(3))


# =========================================================
# 4) Merge comparison pairs
# =========================================================
def merge_pair(left_df, right_df, left_name, right_name):
    left = left_df[[
        "query_key", "seed", "set_id", "target_item_id", "target_item_fg",
        "rank", "hit@10", "top10_ids", "top10_ids_list"
    ]].copy()
    right = right_df[[
        "query_key", "rank", "hit@10", "top10_ids", "top10_ids_list"
    ]].copy()

    merged = (
        left
        .merge(right, on="query_key", how="inner",
               suffixes=(f"_{left_name}", f"_{right_name}"))
        .merge(orig_title_df, on="set_id", how="left")
        .merge(frag_df,       on="set_id", how="left")
    )
    return merged


m_orig_full = merge_pair(detail_original,    detail_full, "orig",       "full")
m_weather   = merge_pair(detail_no_weather,  detail_full, "no_weather",  "full")
m_occasion  = merge_pair(detail_no_occasion, detail_full, "no_occasion", "full")
m_style     = merge_pair(detail_no_style,    detail_full, "no_style",    "full")

print("Merged rows:")
for name, df in [("orig→full", m_orig_full), ("weather→full", m_weather),
                 ("occasion→full", m_occasion), ("style→full", m_style)]:
    print(f"  {name:16s}: {len(df)}")


# =========================================================
# 5) 圖片工具
# =========================================================
def find_image_by_id(img_id):
    for root in IMAGE_ROOTS:
        for ext in (".jpg", ".jpeg", ".png", ".webp"):
            p = Path(root) / f"{img_id}{ext}"
            if p.exists():
                return str(p)
    return None


def open_image_or_none(img_id):
    fp = find_image_by_id(str(img_id))
    if fp is None:
        return None
    try:
        return Image.open(fp).convert("RGB")
    except Exception:
        return None


# =========================================================
# 6) Combined aggregation（5 variants 同一 case）
# =========================================================

# 保留舊的 aggregate_case_candidates 供 category/term analysis 使用
GROUP_COLS = [
    "set_id", "target_item_id", "target_item_fg",
    "original_text", "original_title_only", "original_url_part",
    "title_full_text",
    "title_no_weather_text", "title_no_occasion_text", "title_no_style_text",
    "weather_text", "occasion_text", "style_text",
]

def aggregate_case_candidates(df, left_name, right_name):
    out = df.copy()
    out["rank_gain"]      = out[f"rank_{left_name}"] - out[f"rank_{right_name}"]
    out["improved_hit10"] = (
        (out[f"rank_{left_name}"] > 10) & (out[f"rank_{right_name}"] <= 10)
    ).astype(int)
    out["worse"] = (out[f"rank_{right_name}"] > out[f"rank_{left_name}"]).astype(int)
    existing = [c for c in GROUP_COLS if c in out.columns]
    grp = out.groupby(existing, dropna=False).agg(
        n_seeds        =("seed",               "size"),
        mean_rank_left =(f"rank_{left_name}",  "mean"),
        mean_rank_right=(f"rank_{right_name}", "mean"),
        mean_rank_gain =("rank_gain",          "mean"),
        success_shift  =("improved_hit10",     "sum"),
        worse_count    =("worse",              "sum"),
        win_count      =("rank_gain",          lambda s: (s > 0).sum()),
    ).reset_index()
    grp["comparison"] = f"{left_name} -> {right_name}"
    grp["win_rate"]   = grp["win_count"] / grp["n_seeds"]
    return grp.sort_values(
        ["success_shift", "win_rate", "mean_rank_gain"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

agg_orig_full = aggregate_case_candidates(m_orig_full, "orig",       "full")
agg_weather   = aggregate_case_candidates(m_weather,   "no_weather",  "full")
agg_occasion  = aggregate_case_candidates(m_occasion,  "no_occasion", "full")
agg_style     = aggregate_case_candidates(m_style,     "no_style",    "full")

# ── 6a) 建立 combined dataframe（join 全部 4 個 merged df）──
def _slim(df, variant):
    return df[["query_key", f"rank_{variant}", f"top10_ids_list_{variant}"]].copy()

combined = (
    m_orig_full[[
        "query_key", "seed", "set_id", "target_item_id", "target_item_fg",
        "original_text", "title_full_text",
        "title_no_weather_text", "title_no_occasion_text", "title_no_style_text",
        "weather_text", "occasion_text", "style_text",
        "rank_orig", "rank_full",
        "top10_ids_list_orig", "top10_ids_list_full",
        "hit@10_orig", "hit@10_full",
    ]]
    .merge(_slim(m_weather,  "no_weather"),  on="query_key", how="left")
    .merge(_slim(m_occasion, "no_occasion"), on="query_key", how="left")
    .merge(_slim(m_style,    "no_style"),    on="query_key", how="left")
)

# ── 6b) 聚合：跨 seed 取平均 rank ────────────────────────
COMB_GROUP_COLS = [
    "set_id", "target_item_id", "target_item_fg",
    "original_text", "title_full_text",
    "title_no_weather_text", "title_no_occasion_text", "title_no_style_text",
    "weather_text", "occasion_text", "style_text",
]
existing_comb = [c for c in COMB_GROUP_COLS if c in combined.columns]

agg_combined = combined.groupby(existing_comb, dropna=False).agg(
    n_seeds          =("seed",           "size"),
    mean_rank_orig   =("rank_orig",      "mean"),
    mean_rank_full   =("rank_full",      "mean"),
    mean_rank_weather=("rank_no_weather","mean"),
    mean_rank_occasion=("rank_no_occasion","mean"),
    mean_rank_style  =("rank_no_style",  "mean"),
    hit10_orig       =("hit@10_orig",    "mean"),
    hit10_full       =("hit@10_full",    "mean"),
).reset_index()

# ── 6c) 嚴格過濾：full rank < orig rank ─────────────────
agg_combined = agg_combined[
    agg_combined["mean_rank_full"] < agg_combined["mean_rank_orig"]
].copy()

agg_combined["rank_gain"]      = agg_combined["mean_rank_orig"] - agg_combined["mean_rank_full"]
agg_combined["improved_hit10"] = (
    (agg_combined["hit10_orig"] < 0.5) & (agg_combined["hit10_full"] >= 0.5)
).astype(int)
agg_combined["category_name"]  = agg_combined["target_item_fg"].apply(map_category)

agg_combined = agg_combined.sort_values(
    ["improved_hit10", "rank_gain"], ascending=[False, False]
).reset_index(drop=True)

print(f"\n✅ Combined agg: {len(agg_combined)} valid cases (full < orig)")
display(agg_combined[[
    "set_id", "target_item_id", "category_name",
    "mean_rank_orig", "mean_rank_weather", "mean_rank_occasion",
    "mean_rank_style", "mean_rank_full", "rank_gain"
]].head(10))


# =========================================================
# 7) 論文用個案展示（5 行：Orig / No-W / No-Occ / No-Style / Full）
# =========================================================
def show_full_ablation_case(agg_df, case_idx=0, topk=5, save_fig=False):
    """
    一張圖展示同一 case 的全部 5 個 variant：
      Row 0  Query Target header
      Row 1  Target image（置中）
      Row 2  Original (Baseline) header
      Row 3  Original top-k
      Row 4  No-Weather header
      Row 5  No-Weather top-k
      Row 6  No-Occasion header
      Row 7  No-Occasion top-k
      Row 8  No-Style header
      Row 9  No-Style top-k
      Row 10 Full (Proposed) header
      Row 11 Full top-k
    過濾條件：full rank < orig rank（已在 agg_combined 中保證）
    """
    if case_idx >= len(agg_df):
        print(f"case_idx={case_idx} out of range (total={len(agg_df)})")
        return

    case     = agg_df.iloc[case_idx]
    sid      = str(case["set_id"])
    tid      = str(case["target_item_id"])
    cat_name = case["category_name"]

    # 先取 mean rank（用於 header 顯示）
    mr_orig    = case["mean_rank_orig"]
    mr_weather = case["mean_rank_weather"]
    mr_occasion= case["mean_rank_occasion"]
    mr_style   = case["mean_rank_style"]
    mr_full    = case["mean_rank_full"]

    # 選代表 seed：rank_full 最接近 mean_rank_full 的那個
    # 這樣圖片位置和 header 顯示的 mean rank 最吻合
    cand = combined[
        (combined["set_id"].astype(str)         == sid) &
        (combined["target_item_id"].astype(str) == tid)
    ].copy()

    if cand.empty:
        print("No raw rows found.")
        return

    cand["_dist"] = (cand["rank_full"] - mr_full).abs()
    row = cand.sort_values("_dist").iloc[0]

    def _ids(col):
        v = row.get(col)
        return (v if isinstance(v, list) else [])[:topk]

    ids_orig     = _ids("top10_ids_list_orig")
    ids_weather  = _ids("top10_ids_list_no_weather")
    ids_occasion = _ids("top10_ids_list_no_occasion")
    ids_style    = _ids("top10_ids_list_no_style")
    ids_full     = _ids("top10_ids_list_full")

    # Header/圖片 rank：用代表 seed 的實際 rank（整數，與圖片完全對應）
    r_orig     = int(row["rank_orig"])
    r_weather  = int(row["rank_no_weather"])  if pd.notna(row.get("rank_no_weather"))  else None
    r_occasion = int(row["rank_no_occasion"]) if pd.notna(row.get("rank_no_occasion")) else None
    r_style    = int(row["rank_no_style"])    if pd.notna(row.get("rank_no_style"))    else None
    r_full     = int(row["rank_full"])

    # ── 文字摘要（同時顯示代表 seed rank 和 mean rank）────
    SEP  = "=" * 72
    DASH = "─" * 72
    print(f"\n{SEP}")
    print(f"  Case #{case_idx}  |  set={sid}  |  target={tid}  |  [{cat_name}]")
    print(DASH)
    print(f"  {'variant':>12s}  {'seed rank':>10s}  {'mean rank':>10s}")
    print(DASH)
    for lbl, rk, mrk in [
        ("orig",       r_orig,    mr_orig),
        ("no_weather", r_weather, mr_weather),
        ("no_occasion",r_occasion,mr_occasion),
        ("no_style",   r_style,   mr_style),
        ("full",       r_full,    mr_full),
    ]:
        if rk is None:
            print(f"  {lbl:>12s}  {'—':>10s}  {mrk:>10.1f}")
            continue
        arrow = "—" if lbl == "orig" else ("▲" if rk < r_orig else "▼")
        print(f"  {lbl:>12s}  {rk:>10d}  {mrk:>10.1f}  {arrow}")
    print(DASH)
    print(f"  [Original] {str(case.get('original_text',   ''))[:110]}")
    print(f"  [Full    ] {str(case.get('title_full_text', ''))[:110]}")
    print(f"  ↳ Weather : {case.get('weather_text','')}")
    print(f"  ↳ Occasion: {case.get('occasion_text','')}")
    print(f"  ↳ Style   : {case.get('style_text','')}")
    print(f"{SEP}\n")

    # ── result rows ──────────────────────────────────────
    result_rows = [
        (ids_orig,     r_orig,    "Original (Baseline)",    "#555555"),
        (ids_weather,  r_weather, "No-Weather (ablation)",  "#C0392B"),
        (ids_occasion, r_occasion,"No-Occasion (ablation)", "#E67E22"),
        (ids_style,    r_style,   "No-Style (ablation)",    "#8E44AD"),
        (ids_full,     r_full,    "Full (Proposed)",         "#2166AC"),
    ]
    n_result = len(result_rows)  # 5

    # ── GridSpec ─────────────────────────────────────────
    from matplotlib.gridspec import GridSpec
    HDR_H, IMG_H, TGT_H = 0.30, 2.5, 2.2
    height_ratios = [HDR_H, TGT_H] + [HDR_H, IMG_H] * n_result
    n_gs = 2 + n_result * 2

    fig_w = max(topk, 3) * 2.45 + 0.3
    fig_h = sum(height_ratios) + 0.5

    fig = plt.figure(figsize=(fig_w, fig_h))
    gs  = GridSpec(n_gs, topk, figure=fig,
                   height_ratios=height_ratios,
                   hspace=0.0, wspace=0.07,
                   top=0.97, bottom=0.03, left=0.02, right=0.98)

    def _draw(ax, img_id, is_hit=False, is_target=False):
        img = open_image_or_none(img_id)
        if img is not None:
            ax.imshow(img)
        else:
            ax.set_facecolor("#eeeeee")
            ax.text(0.5, 0.5, "N/A", ha="center", va="center",
                    fontsize=7, color="#aaaaaa", transform=ax.transAxes)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel(
            "✓ Hit" if is_hit else "",
            fontsize=8, color="#27AE60" if is_hit else "#666666",
            fontweight="bold" if is_hit else "normal", labelpad=2
        )
        if is_target: lw, ec = 4.0, "#111111"
        elif is_hit:  lw, ec = 5.5, "#27AE60"
        else:         lw, ec = 0.7, "#cccccc"
        for sp in ax.spines.values():
            sp.set_visible(True); sp.set_edgecolor(ec); sp.set_linewidth(lw)

    def _header(gs_row, text, color):
        ax_h = fig.add_subplot(gs[gs_row, :])
        ax_h.axis("off")
        ax_h.text(0.5, 0.45, text, ha="center", va="center",
                  fontsize=10, fontweight="bold", color=color,
                  transform=ax_h.transAxes)

    # Row 0: Target header
    _header(0, f"Query Target  —  {cat_name}", "#333333")
    # Row 1: Target image (centered)
    mid = topk // 2
    for c in range(topk):
        ax = fig.add_subplot(gs[1, c])
        if c == mid:
            _draw(ax, tid, is_target=True)
        else:
            ax.set_visible(False)

    # Row 2…: result rows
    for r, (ids, rank, label, color) in enumerate(result_rows):
        rank_str = str(rank) if rank is not None else "—"
        _header(2 + r * 2, f"{label}   (rank = {rank_str})", color)
        for i in range(topk):
            ax = fig.add_subplot(gs[2 + r * 2 + 1, i])
            if i < len(ids):
                iid = ids[i]
                _draw(ax, iid, is_hit=(str(iid) == str(tid)))
                ax.text(0.03, 0.97, f"#{i+1}", transform=ax.transAxes,
                        fontsize=7.5, va="top", ha="left", color="#444444",
                        bbox=dict(fc="white", ec="none", alpha=0.7, pad=1))
            else:
                ax.set_visible(False)

    fig.legend(
        handles=[mpatches.Patch(facecolor="#27AE60", edgecolor="#27AE60",
                                label="Ground-truth target retrieved")],
        loc="lower center", ncol=1, fontsize=8,
        frameon=False, bbox_to_anchor=(0.5, 0.0)
    )
    fig.suptitle(
        f"Case #{case_idx}  |  set={sid}  |  {cat_name}",
        fontsize=10, fontweight="bold", y=0.99
    )

    if SAVE_FIGS or save_fig:
        fname = f"full_ablation_case_idx{case_idx:02d}.pdf"
        fpath = os.path.join(FIG_DIR, fname)
        plt.savefig(fpath, dpi=300, bbox_inches="tight")
        print(f"[saved] {fpath}")
    plt.show()


# =========================================================
# 8) Category analysis
# =========================================================
def category_summary(df, left_name, right_name):
    grp = df.groupby("target_item_fg").agg(
        n          =("query_key",          "size"),
        left_hit10 =(f"hit@10_{left_name}",  "mean"),
        right_hit10=(f"hit@10_{right_name}", "mean"),
        left_rank  =(f"rank_{left_name}",    "median"),
        right_rank =(f"rank_{right_name}",   "median"),
    ).reset_index()
    grp["delta_hit10"] = grp["right_hit10"] - grp["left_hit10"]
    grp["delta_rank"]  = grp["right_rank"]  - grp["left_rank"]
    # 套用名稱
    grp["category_name"] = grp["target_item_fg"].apply(map_category)
    return grp.sort_values(
        ["delta_hit10", "n"], ascending=[False, False]
    ).reset_index(drop=True)


cat_orig_full = category_summary(m_orig_full, "orig",       "full")
cat_weather   = category_summary(m_weather,   "no_weather",  "full")
cat_occasion  = category_summary(m_occasion,  "no_occasion", "full")
cat_style     = category_summary(m_style,     "no_style",    "full")

print("\n=== Category analysis: Original → Proposed ===")
display(cat_orig_full[["category_name", "n", "left_hit10", "right_hit10", "delta_hit10"]].head(10))
print("\n=== Category analysis: Weather contribution ===")
display(cat_weather  [["category_name", "n", "left_hit10", "right_hit10", "delta_hit10"]].head(10))
print("\n=== Category analysis: Occasion contribution ===")
display(cat_occasion [["category_name", "n", "left_hit10", "right_hit10", "delta_hit10"]].head(10))
print("\n=== Category analysis: Style contribution ===")
display(cat_style    [["category_name", "n", "left_hit10", "right_hit10", "delta_hit10"]].head(10))


# =========================================================
# 9) 論文用 Category Bar Chart
# =========================================================
def plot_paper_category_bar(
    df, title,
    label_col="category_name", value_col="delta_hit10",
    top_n=8, save_name=None,
    color_pos="#2166AC", color_neg="#D6604D",
    figsize=(7.4, 4.2)
):
    plot_df = df.head(top_n).copy().sort_values(value_col, ascending=True)
    vals = plot_df[value_col].astype(float).to_numpy()
    colors = [color_pos if v >= 0 else color_neg for v in vals]

    fig, ax = plt.subplots(figsize=figsize)

    bars = ax.barh(
        plot_df[label_col].astype(str),
        plot_df[value_col],
        color=colors, edgecolor="white", height=0.6
    )

    # --- 留出左右空間，避免數字被擠到邊界 ---
    max_abs = np.abs(vals).max() if len(vals) else 0.001
    text_pad = max(max_abs * 0.02, 0.00015)
    x_pad    = max(max_abs * 0.18, 0.0005)

    xmin = min(vals.min(), 0) - x_pad
    xmax = max(vals.max(), 0) + x_pad
    ax.set_xlim(xmin, xmax)

    # --- 數值標籤 ---
    for bar, val in zip(bars, vals):
        y = bar.get_y() + bar.get_height() / 2

        if val >= 0:
            # 正值：放在 bar 右邊
            x = val + text_pad
            ha = "left"
        else:
            # 負值：改放在 bar 內側/右側，避免往左壓到類別文字
            x = val + text_pad
            ha = "left"

        ax.text(
            x, y, f"{val:+.3f}",
            va="center", ha=ha, fontsize=8.5
        )

    ax.axvline(0, color="gray", linestyle="--", linewidth=0.9)
    ax.set_xlabel("ΔHit@10  (Proposed − Baseline)", fontsize=10)
    ax.set_title(title, fontsize=11, fontweight="bold", pad=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="y", labelsize=9)

    # 左邊再多留一點空間給長類別名稱
    fig.subplots_adjust(left=0.36, right=0.97, top=0.88, bottom=0.14)

    if SAVE_FIGS and save_name:
        fpath = os.path.join(FIG_DIR, save_name)
        plt.savefig(fpath, dpi=300, bbox_inches="tight")
        print(f"[saved] {fpath}")

    plt.show()


plot_paper_category_bar(
    cat_orig_full,
    title="Top Categories Improved by Proposed Description",
    top_n=8, save_name="fig_category_orig_vs_full.pdf"
)
plot_paper_category_bar(
    cat_weather,
    title="Categories Most Sensitive to Weather Information",
    top_n=8, save_name="fig_category_weather.pdf"
)
plot_paper_category_bar(
    cat_occasion,
    title="Categories Most Sensitive to Occasion Information",
    top_n=8, save_name="fig_category_occasion.pdf"
)
plot_paper_category_bar(
    cat_style,
    title="Categories Most Sensitive to Style Information",
    top_n=8, save_name="fig_category_style.pdf"
)


# =========================================================
# 10) Condition / Term analysis
# =========================================================
def explode_terms(df, terms_col):
    tmp = df.copy()
    tmp[terms_col] = tmp[terms_col].apply(
        lambda x: x if isinstance(x, list) else []
    )
    tmp = tmp.explode(terms_col)
    return tmp[
        tmp[terms_col].notna() &
        (tmp[terms_col].astype(str).str.strip() != "")
    ]


def term_summary(df, terms_col, left_name, right_name, min_n=20):
    tmp = explode_terms(df, terms_col)
    grp = tmp.groupby(terms_col).agg(
        n          =("query_key",          "size"),
        left_hit10 =(f"hit@10_{left_name}",  "mean"),
        right_hit10=(f"hit@10_{right_name}", "mean"),
        left_rank  =(f"rank_{left_name}",    "median"),
        right_rank =(f"rank_{right_name}",   "median"),
    ).reset_index()
    grp["delta_hit10"] = grp["right_hit10"] - grp["left_hit10"]
    grp["delta_rank"]  = grp["right_rank"]  - grp["left_rank"]
    return (
        grp[grp["n"] >= min_n]
        .sort_values(["delta_hit10", "n"], ascending=[False, False])
        .reset_index(drop=True)
    )


weather_terms_orig_full  = term_summary(m_orig_full, "weather_terms",  "orig",       "full")
occasion_terms_orig_full = term_summary(m_orig_full, "occasion_terms", "orig",       "full")
style_terms_orig_full    = term_summary(m_orig_full, "style_terms",    "orig",       "full")
weather_terms_factor     = term_summary(m_weather,   "weather_terms",  "no_weather",  "full")
occasion_terms_factor    = term_summary(m_occasion,  "occasion_terms", "no_occasion", "full")
style_terms_factor       = term_summary(m_style,     "style_terms",    "no_style",    "full")

print("\n=== Weather  terms (orig → full) ==="); display(weather_terms_orig_full.head(10))
print("\n=== Occasion terms (orig → full) ==="); display(occasion_terms_orig_full.head(10))
print("\n=== Style    terms (orig → full) ==="); display(style_terms_orig_full.head(10))
print("\n=== Weather  contribution ===");         display(weather_terms_factor.head(10))
print("\n=== Occasion contribution ===");         display(occasion_terms_factor.head(10))
print("\n=== Style    contribution ===");         display(style_terms_factor.head(10))


# Term bar charts（僅在有足夠資料時）
for term_df, col, title, fname in [
    (weather_terms_orig_full,  "weather_terms",
     "Top Weather Terms Improved by Proposed Description",
     "fig_weather_terms_orig_vs_full.pdf"),
    (occasion_terms_orig_full, "occasion_terms",
     "Top Occasion Terms Improved by Proposed Description",
     "fig_occasion_terms_orig_vs_full.pdf"),
    (style_terms_orig_full,    "style_terms",
     "Top Style Terms Improved by Proposed Description",
     "fig_style_terms_orig_vs_full.pdf"),
    (weather_terms_factor,     "weather_terms",
     "Weather Term Contribution (no_weather vs full)",
     "fig_weather_terms_factor.pdf"),
    (occasion_terms_factor,    "occasion_terms",
     "Occasion Term Contribution (no_occasion vs full)",
     "fig_occasion_terms_factor.pdf"),
    (style_terms_factor,       "style_terms",
     "Style Term Contribution (no_style vs full)",
     "fig_style_terms_factor.pdf"),
]:
    if len(term_df) > 0:
        plot_paper_category_bar(
            term_df, title=title,
            label_col=col, value_col="delta_hit10",
            top_n=8, save_name=fname,
            figsize=(7, 4)
        )


# =========================================================
# 11) Failure cases
# =========================================================
FAIL_GROUP_COLS = [
    "set_id", "target_item_id", "target_item_fg",
    "original_text", "title_full_text",
    "title_no_weather_text", "title_no_occasion_text", "title_no_style_text",
    "weather_text", "occasion_text", "style_text",
]


def aggregate_failure_candidates(df, left_name, right_name):
    out = df.copy()
    out["rank_loss"]   = out[f"rank_{right_name}"] - out[f"rank_{left_name}"]
    out["became_worse"] = (out[f"rank_{right_name}"] > out[f"rank_{left_name}"]).astype(int)

    existing = [c for c in FAIL_GROUP_COLS if c in out.columns]
    grp = out.groupby(existing, dropna=False).agg(
        n_seeds        =("seed",          "size"),
        mean_rank_left =(f"rank_{left_name}",  "mean"),
        mean_rank_right=(f"rank_{right_name}", "mean"),
        mean_rank_loss =("rank_loss",      "mean"),
        worse_count    =("became_worse",   "sum"),
    ).reset_index()

    grp["comparison"] = f"{left_name} -> {right_name}"
    grp["worse_rate"] = grp["worse_count"] / grp["n_seeds"]
    grp["category_name"] = grp["target_item_fg"].apply(map_category)

    return grp.sort_values(
        ["worse_rate", "mean_rank_loss"], ascending=[False, False]
    ).reset_index(drop=True)


fail_orig_full = aggregate_failure_candidates(m_orig_full, "orig",       "full")
fail_weather   = aggregate_failure_candidates(m_weather,   "no_weather",  "full")
fail_occasion  = aggregate_failure_candidates(m_occasion,  "no_occasion", "full")
fail_style     = aggregate_failure_candidates(m_style,     "no_style",    "full")

print("\n=== Failure cases: Original → Proposed ===")
display(fail_orig_full[["set_id","target_item_id","category_name",
                         "mean_rank_left","mean_rank_right","worse_rate"]].head(10))
print("\n=== Failure cases: Weather contribution ===")
display(fail_weather  [["set_id","target_item_id","category_name",
                         "mean_rank_left","mean_rank_right","worse_rate"]].head(10))
print("\n=== Failure cases: Occasion contribution ===")
display(fail_occasion [["set_id","target_item_id","category_name",
                         "mean_rank_left","mean_rank_right","worse_rate"]].head(10))
print("\n=== Failure cases: Style contribution ===")
display(fail_style    [["set_id","target_item_id","category_name",
                         "mean_rank_left","mean_rank_right","worse_rate"]].head(10))


# =========================================================
# 12) 儲存最終摘要表
# =========================================================
if SAVE_TABLES:
    pd.concat([
        agg_orig_full.assign(case_type="orig_to_full"),
        agg_weather  .assign(case_type="weather"),
        agg_occasion .assign(case_type="occasion"),
        agg_style    .assign(case_type="style"),
    ]).to_csv(os.path.join(TABLE_DIR, "qual_user_cases.csv"), index=False)

    pd.concat([
        fail_orig_full.assign(case_type="orig_to_full"),
        fail_weather  .assign(case_type="weather"),
        fail_occasion .assign(case_type="occasion"),
        fail_style    .assign(case_type="style"),
    ]).to_csv(os.path.join(TABLE_DIR, "qual_failure_cases.csv"), index=False)

    pd.concat([
        cat_orig_full.assign(case_type="orig_to_full"),
        cat_weather  .assign(case_type="weather"),
        cat_occasion .assign(case_type="occasion"),
        cat_style    .assign(case_type="style"),
    ]).to_csv(os.path.join(TABLE_DIR, "qual_category_summary.csv"), index=False)

    pd.concat([
        weather_terms_orig_full .assign(case_type="weather_orig_to_full"),
        occasion_terms_orig_full.assign(case_type="occasion_orig_to_full"),
        style_terms_orig_full   .assign(case_type="style_orig_to_full"),
        weather_terms_factor    .assign(case_type="weather_factor"),
        occasion_terms_factor   .assign(case_type="occasion_factor"),
        style_terms_factor      .assign(case_type="style_factor"),
    ]).to_csv(os.path.join(TABLE_DIR, "qual_condition_summary.csv"), index=False)

    print(f"[saved] tables → {TABLE_DIR}")


# =========================================================
# 13) 自動顯示最佳 Top-N 個案（5 行全ablation版）
# =========================================================
TOP_N_CASES = 5   # 每組顯示幾個 case，可自行調整

print(f"\n{'#'*68}")
print(f"  ▶  Full Ablation Case Study  —  Top-{TOP_N_CASES} Best Cases")
print(f"     篩選條件：full rank < orig rank（共 {len(agg_combined)} 筆）")
print(f"{'#'*68}")

for idx in range(min(TOP_N_CASES, len(agg_combined))):
    show_full_ablation_case(agg_combined, case_idx=idx, topk=TOPK)

print("\n✅ 全部個案展示完成")

[categories.csv] 載入 159 個類別，範例：{'2': 'undefined', '3': 'dress (all-body)', '4': 'dress (all-body)', '5': 'dress (all-body)'}
Loaded rows:
  original    : 17160
  full        : 17160
  no_weather  : 17160
  no_occasion : 17160
  no_style    : 17160
Loaded original titles : 68306


,set_id,original_text,original_title_only,original_url_part
0,219779031,spring trend colored denim Denim Jumpers,Denim Jumpers,spring trend colored denim
1,211158670,boot scootin outfit only 2500 Stacked-Heel Boots,Stacked-Heel Boots,boot scootin outfit only 2500
2,222049137,"mermaid for life ""Mermaid Hair, Don't Care""","""Mermaid Hair, Don't Care""",mermaid for life


Loaded fragments       : 35140


,set_id,title_full_text,weather_terms,occasion_terms,style_terms,weather_text,occasion_text,style_text,title_no_weather_text,title_no_occasion_text,title_no_style_text
0,201033839,"19°C, Effortless Mint & Lace Ensemble for a Br...","[19°C, Breezy Spring Day]",[],[Effortless Mint & Lace Ensemble],19°C | Breezy Spring Day,,Effortless Mint & Lace Ensemble,Effortless Mint & Lace Ensemble,"19°C, Effortless Mint & Lace Ensemble Breezy S...","19°C, Breezy Spring Day"
1,205002070,"24°C, Effortless Boho Vibes for Warm Days","[24°C, Warm Days]",[],[Effortless Boho Vibes],24°C | Warm Days,,Effortless Boho Vibes,Effortless Boho Vibes,"24°C, Effortless Boho Vibes Warm Days","24°C, Warm Days"
2,217778087,"17°C, Effortless Evening Glam with a Cozy Flar...",[17°C],[Evening],"[Effortless, Glam, Cozy, Flare Coat]",17°C,Evening,Effortless | Glam | Cozy | Flare Coat,Effortless Evening Glam Cozy Flare Coat,"17°C, Effortless Glam Cozy Flare Coat","17°C, Evening"


Merged rows:
  orig→full       : 17160
  weather→full    : 17160
  occasion→full   : 17160
  style→full      : 17160

✅ Combined agg: 1932 valid cases (full < orig)


...


  Case #3  |  set=194817964  |  target=167658682  |  [dress (all-body)]
────────────────────────────────────────────────────────────────────────
       variant   seed rank   mean rank
────────────────────────────────────────────────────────────────────────
          orig         124       214.2  —
    no_weather           3         3.2  ▲
   no_occasion           6         8.0  ▲
      no_style          32       138.2  ▲
          full           4         4.0  ▲
────────────────────────────────────────────────────────────────────────
  [Original] spring formal
  [Full    ] 22.3° C, Retro Picnic Chic – A playful red and white gingham dress with heels and sunglasses for a warm-weathe
  ↳ Weather : 22.3° C | warm-weather
  ↳ Occasion: celebration
  ↳ Style   : Retro Picnic Chic | playful red and white gingham dress | heels | sunglasses



<Figure size 2008x2720 with 36 Axes>


  Case #4  |  set=30160190  |  target=12185913  |  [purse (bags)]
────────────────────────────────────────────────────────────────────────
       variant   seed rank   mean rank
────────────────────────────────────────────────────────────────────────
          orig         691       192.4  —
    no_weather          20        18.4  ▲
   no_occasion          29        37.0  ▲
      no_style         106        36.4  ▲
          full           6        14.8  ▲
────────────────────────────────────────────────────────────────────────
  [Original] morning commute Morning Commute
  [Full    ] 23.9° C, Velvet Evening Commute Style
  ↳ Weather : 23.9° C
  ↳ Occasion: Evening Commute
  ↳ Style   : Velvet | Style



<Figure size 2008x2720 with 36 Axes>


✅ 全部個案展示完成


In [2]:
import pandas as pd

old = pd.read_csv("./experiments/detail_cir_old_seed1.csv")
new = pd.read_csv("./experiments/detail_cir_new_seed1.csv")

print("OLD Recall@10:", old["hit@10"].mean(), "Median rank:", old["rank"].median())
print("NEW Recall@10:", new["hit@10"].mean(), "Median rank:", new["rank"].median())

cat_old = old.groupby("target_item_fg").agg(n=("rank","size"), R10_old=("hit@10","mean"), med_old=("rank","median"))
cat_new = new.groupby("target_item_fg").agg(R10_new=("hit@10","mean"), med_new=("rank","median"))
cat = cat_old.join(cat_new, how="inner")
cat["delta_R10"] = cat["R10_new"] - cat["R10_old"]
cat["delta_med_rank"] = cat["med_new"] - cat["med_old"]
print(cat.sort_values("delta_R10", ascending=False).head(20))

OLD Recall@10: 0.07797229083879283 Median rank: 230.0
NEW Recall@10: 0.0869938782085705 Median rank: 205.0
                  n   R10_old  med_old   R10_new  med_new  delta_R10  \
target_item_fg                                                         
46              795  0.091824    181.0  0.113208    148.0   0.021384   
9               295  0.084746    235.0  0.105085    160.0   0.020339   
318             466  0.079399    241.0  0.096567    237.5   0.017167   
57              488  0.051230    293.0  0.067623    307.5   0.016393   
11              490  0.102041    192.0  0.118367    174.0   0.016327   
19              371  0.053908    280.0  0.070081    253.0   0.016173   
36              402  0.067164    199.5  0.079602    176.5   0.012438   
64              662  0.064955    251.5  0.077039    214.0   0.012085   
106             397  0.068010    265.0  0.078086    270.0   0.010076   
38              520  0.067308    221.5  0.076923    223.5   0.009615   
4               752  0.107713

In [26]:
import os
import re
import json
import glob
import ast
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from IPython.display import display

# =========================================================
# 0) CONFIG
# =========================================================
EXP_ROOT = "./experiments_none_only"
DETAIL_DIR = os.path.join(EXP_ROOT, "detail_cir")
if not os.path.exists(DETAIL_DIR):
    DETAIL_DIR = EXP_ROOT

DETAIL_RUN_TAG    = "none_subset"
DETAIL_SUBSET_TAG = "subset"

# 新描述 / fragments / ablation 來源
FRAG_JSONL = "/home/ester/valid_description/wos_split_results_v5_merged_retry_round3.jsonl"

# 原始描述來源
ORIGINAL_TITLE_JSON = "/home/ester/valid_description/polyvore_data/polyvore_outfits/polyvore_outfit_titles.json"

# Category mapping CSV
CATEGORY_CSV = "/home/ester/valid_description/polyvore_data/polyvore_outfits/categories.csv"

# 圖片路徑
IMAGE_ROOTS = [
    "/home/ester/valid_description/polyvore_data/polyvore_outfits/images",
]

SAVE_TABLES = False
SAVE_FIGS   = False      # True → 存 PDF/PNG

OUTPUT_DIR = "./qualitative_outputs_final"
TABLE_DIR  = os.path.join(OUTPUT_DIR, "tables")
FIG_DIR    = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TABLE_DIR,  exist_ok=True)
os.makedirs(FIG_DIR,    exist_ok=True)

VARIANTS = {
    "original"   : "outfitUrlTitle",
    "full"       : "NewoutfitUrlTitle",
    "no_weather" : "no_weather",
    "no_occasion": "no_occasion",
    "no_style"   : "no_style",
}

TOPK = 5

plt.rcParams.update({
    "figure.dpi"         : 160,
    "font.size"          : 10,
    "axes.spines.top"    : False,
    "axes.spines.right"  : False,
    "font.family"        : "DejaVu Sans",
})


# =========================================================
# 0b) Category mapping
# =========================================================
def load_category_map(csv_path):
    """
    讀取 categories.csv。
    支援格式：col0=id, col1=name, col2=group（無 header）
    回傳 dict: {str(id): "name (group)"}  例如 "dress (all-body)"
    若無 group 欄則僅回傳 name。
    同一 id 可能對應多個 name（取第一筆）。
    """
    if not os.path.exists(csv_path):
        print(f"[WARNING] 找不到 categories.csv：{csv_path}")
        print("         → category 欄位將顯示原始數字 ID")
        return {}

    # 先嘗試無 header 讀取，若第一欄為數字即為無 header 格式
    df_raw = pd.read_csv(csv_path, header=None)
    first_val = str(df_raw.iloc[0, 0]).strip()
    if first_val.isdigit():
        df = df_raw.copy()
    else:
        # 有 header：自動偵測欄位
        df = pd.read_csv(csv_path)
        id_candidates   = ["id", "category_id", "fg"]
        name_candidates = ["name", "category_name", "category"]
        grp_candidates  = ["group", "supercategory", "type", "parent"]
        id_col  = next((c for c in id_candidates   if c in df.columns), df.columns[0])
        nm_col  = next((c for c in name_candidates if c in df.columns), df.columns[1])
        grp_col = next((c for c in grp_candidates  if c in df.columns), None)
        df = df.rename(columns={id_col: 0, nm_col: 1})
        if grp_col:
            df = df.rename(columns={grp_col: 2})

    has_group = df.shape[1] >= 3

    result = {}
    for _, row in df.iterrows():
        cid = str(row[0]).strip()
        if cid in result:          # 同 id 取第一筆
            continue
        name = str(row[1]).strip() if pd.notna(row[1]) else cid
        if has_group and pd.notna(row[2]):
            grp   = str(row[2]).strip()
            label = f"{name} ({grp})"
        else:
            label = name
        result[cid] = label

    sample = dict(list(result.items())[:4])
    print(f"[categories.csv] 載入 {len(result)} 個類別，範例：{sample}")
    return result


CATEGORY_NAME_MAP = load_category_map(CATEGORY_CSV)


def map_category(x):
    """數字 → 類別名稱；找不到就直接回傳原始字串。"""
    return CATEGORY_NAME_MAP.get(str(x), str(x))


# =========================================================
# 1) 讀取 detail csv
# =========================================================
def load_one_variant_detail(variant_name):
    pattern = os.path.join(
        DETAIL_DIR,
        f"detail_cir_{variant_name}_{DETAIL_SUBSET_TAG}_{DETAIL_RUN_TAG}_seed*.csv"
    )
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"找不到 detail 檔案：{pattern}")

    dfs = []
    for fp in files:
        df = pd.read_csv(fp)
        df["variant"]     = variant_name
        df["source_file"] = os.path.basename(fp)
        dfs.append(df)

    return pd.concat(dfs, axis=0, ignore_index=True)


detail_original   = load_one_variant_detail(VARIANTS["original"])
detail_full       = load_one_variant_detail(VARIANTS["full"])
detail_no_weather = load_one_variant_detail(VARIANTS["no_weather"])
detail_no_occasion= load_one_variant_detail(VARIANTS["no_occasion"])
detail_no_style   = load_one_variant_detail(VARIANTS["no_style"])

print("Loaded rows:")
for name, df in [("original", detail_original), ("full", detail_full),
                 ("no_weather", detail_no_weather), ("no_occasion", detail_no_occasion),
                 ("no_style", detail_no_style)]:
    print(f"  {name:12s}: {len(df)}")


# =========================================================
# 2) Schema 正規化
# =========================================================
def normalize_detail_df(df):
    out = df.copy()
    required = ["seed", "set_id", "target_item_id", "target_item_fg",
                "rank", "hit@10", "top10_ids"]
    for c in required:
        if c not in out.columns:
            raise KeyError(f"detail csv 缺少必要欄位：{c}")

    out["seed"]           = out["seed"].astype(int)
    out["set_id"]         = out["set_id"].astype(str)
    out["target_item_id"] = out["target_item_id"].astype(str)
    out["target_item_fg"] = out["target_item_fg"].astype(str)
    out["rank"]           = pd.to_numeric(out["rank"],    errors="coerce")
    out["hit@10"]         = pd.to_numeric(out["hit@10"],  errors="coerce")

    def parse_top10(x):
        if pd.isna(x):
            return []
        if isinstance(x, list):
            return [str(i) for i in x]
        try:
            return [str(i) for i in json.loads(x)]
        except Exception:
            try:
                v = ast.literal_eval(x)
                return [str(i) for i in v] if isinstance(v, list) else []
            except Exception:
                return []

    out["top10_ids_list"] = out["top10_ids"].apply(parse_top10)
    out["query_key"] = (
        out["seed"].astype(str) + "||" +
        out["set_id"]           + "||" +
        out["target_item_id"]
    )
    return out


detail_original    = normalize_detail_df(detail_original)
detail_full        = normalize_detail_df(detail_full)
detail_no_weather  = normalize_detail_df(detail_no_weather)
detail_no_occasion = normalize_detail_df(detail_no_occasion)
detail_no_style    = normalize_detail_df(detail_no_style)


# =========================================================
# 3) 載入 metadata
# =========================================================
def normalize_text(x):
    if x is None:
        return ""
    if isinstance(x, str):
        return x.strip()
    if isinstance(x, list):
        return " | ".join(str(i).strip() for i in x if str(i).strip())
    return str(x).strip()


def load_original_titles_json(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"找不到原始描述檔案：{path}")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []
    for sid, value in data.items():
        original_text = original_title_only = original_url_part = ""
        if isinstance(value, str):
            original_text = value.strip()
        elif isinstance(value, dict):
            title_part = normalize_text(value.get("title", ""))
            url_part   = normalize_text(value.get("url_name", value.get("url", "")))
            original_text      = " ".join(p for p in [url_part, title_part] if p).strip()
            original_title_only = title_part
            original_url_part   = url_part
        else:
            original_text = normalize_text(value)

        rows.append({
            "set_id"            : str(sid),
            "original_text"     : original_text,
            "original_title_only": original_title_only,
            "original_url_part" : original_url_part,
        })
    return pd.DataFrame(rows)


def load_fragments_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"找不到 fragments 檔案：{path}")
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if obj.get("valid") is not True:
                continue

            frags          = obj.get("fragments", {})
            title_ablation = obj.get("title_ablation", {})
            weather_terms  = frags.get("weather",  [])
            occasion_terms = frags.get("occasion", [])
            style_terms    = frags.get("style",    [])

            rows.append({
                "set_id"               : str(obj.get("id", "")),
                "title_full_text"      : normalize_text(obj.get("title", "")),
                "weather_terms"        : weather_terms  if isinstance(weather_terms,  list) else [],
                "occasion_terms"       : occasion_terms if isinstance(occasion_terms, list) else [],
                "style_terms"          : style_terms    if isinstance(style_terms,    list) else [],
                "weather_text"         : normalize_text(weather_terms),
                "occasion_text"        : normalize_text(occasion_terms),
                "style_text"           : normalize_text(style_terms),
                "title_no_weather_text" : normalize_text(title_ablation.get("no_weather",  "")),
                "title_no_occasion_text": normalize_text(title_ablation.get("no_occasion", "")),
                "title_no_style_text"   : normalize_text(title_ablation.get("no_style",    "")),
            })
    return pd.DataFrame(rows)


orig_title_df = load_original_titles_json(ORIGINAL_TITLE_JSON)
frag_df       = load_fragments_jsonl(FRAG_JSONL)

print(f"Loaded original titles : {len(orig_title_df)}")
display(orig_title_df.head(3))
print(f"Loaded fragments       : {len(frag_df)}")
display(frag_df.head(3))


# =========================================================
# 4) Merge comparison pairs
# =========================================================
def merge_pair(left_df, right_df, left_name, right_name):
    left = left_df[[
        "query_key", "seed", "set_id", "target_item_id", "target_item_fg",
        "rank", "hit@10", "top10_ids", "top10_ids_list"
    ]].copy()
    right = right_df[[
        "query_key", "rank", "hit@10", "top10_ids", "top10_ids_list"
    ]].copy()

    merged = (
        left
        .merge(right, on="query_key", how="inner",
               suffixes=(f"_{left_name}", f"_{right_name}"))
        .merge(orig_title_df, on="set_id", how="left")
        .merge(frag_df,       on="set_id", how="left")
    )
    return merged


m_orig_full = merge_pair(detail_original,    detail_full, "orig",       "full")
m_weather   = merge_pair(detail_no_weather,  detail_full, "no_weather",  "full")
m_occasion  = merge_pair(detail_no_occasion, detail_full, "no_occasion", "full")
m_style     = merge_pair(detail_no_style,    detail_full, "no_style",    "full")

print("Merged rows:")
for name, df in [("orig→full", m_orig_full), ("weather→full", m_weather),
                 ("occasion→full", m_occasion), ("style→full", m_style)]:
    print(f"  {name:16s}: {len(df)}")


# =========================================================
# 5) 圖片工具
# =========================================================
def find_image_by_id(img_id):
    for root in IMAGE_ROOTS:
        for ext in (".jpg", ".jpeg", ".png", ".webp"):
            p = Path(root) / f"{img_id}{ext}"
            if p.exists():
                return str(p)
    return None


def open_image_or_none(img_id):
    fp = find_image_by_id(str(img_id))
    if fp is None:
        return None
    try:
        return Image.open(fp).convert("RGB")
    except Exception:
        return None


# =========================================================
# 6) Combined aggregation（5 variants 同一 case）
# =========================================================

# 保留舊的 aggregate_case_candidates 供 category/term analysis 使用
GROUP_COLS = [
    "set_id", "target_item_id", "target_item_fg",
    "original_text", "original_title_only", "original_url_part",
    "title_full_text",
    "title_no_weather_text", "title_no_occasion_text", "title_no_style_text",
    "weather_text", "occasion_text", "style_text",
]

def aggregate_case_candidates(df, left_name, right_name):
    out = df.copy()
    out["rank_gain"]      = out[f"rank_{left_name}"] - out[f"rank_{right_name}"]
    out["improved_hit10"] = (
        (out[f"rank_{left_name}"] > 10) & (out[f"rank_{right_name}"] <= 10)
    ).astype(int)
    out["worse"] = (out[f"rank_{right_name}"] > out[f"rank_{left_name}"]).astype(int)
    existing = [c for c in GROUP_COLS if c in out.columns]
    grp = out.groupby(existing, dropna=False).agg(
        n_seeds        =("seed",               "size"),
        mean_rank_left =(f"rank_{left_name}",  "mean"),
        mean_rank_right=(f"rank_{right_name}", "mean"),
        mean_rank_gain =("rank_gain",          "mean"),
        success_shift  =("improved_hit10",     "sum"),
        worse_count    =("worse",              "sum"),
        win_count      =("rank_gain",          lambda s: (s > 0).sum()),
    ).reset_index()
    grp["comparison"] = f"{left_name} -> {right_name}"
    grp["win_rate"]   = grp["win_count"] / grp["n_seeds"]
    return grp.sort_values(
        ["success_shift", "win_rate", "mean_rank_gain"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

agg_orig_full = aggregate_case_candidates(m_orig_full, "orig",       "full")
agg_weather   = aggregate_case_candidates(m_weather,   "no_weather",  "full")
agg_occasion  = aggregate_case_candidates(m_occasion,  "no_occasion", "full")
agg_style     = aggregate_case_candidates(m_style,     "no_style",    "full")

# ── 6a) 建立 combined dataframe（join 全部 4 個 merged df）──
def _slim(df, variant):
    return df[["query_key", f"rank_{variant}", f"top10_ids_list_{variant}"]].copy()

combined = (
    m_orig_full[[
        "query_key", "seed", "set_id", "target_item_id", "target_item_fg",
        "original_text", "title_full_text",
        "title_no_weather_text", "title_no_occasion_text", "title_no_style_text",
        "weather_text", "occasion_text", "style_text",
        "rank_orig", "rank_full",
        "top10_ids_list_orig", "top10_ids_list_full",
        "hit@10_orig", "hit@10_full",
    ]]
    .merge(_slim(m_weather,  "no_weather"),  on="query_key", how="left")
    .merge(_slim(m_occasion, "no_occasion"), on="query_key", how="left")
    .merge(_slim(m_style,    "no_style"),    on="query_key", how="left")
)

# ── 6b) 聚合：跨 seed 取平均 rank ────────────────────────
COMB_GROUP_COLS = [
    "set_id", "target_item_id", "target_item_fg",
    "original_text", "title_full_text",
    "title_no_weather_text", "title_no_occasion_text", "title_no_style_text",
    "weather_text", "occasion_text", "style_text",
]
existing_comb = [c for c in COMB_GROUP_COLS if c in combined.columns]

agg_combined = combined.groupby(existing_comb, dropna=False).agg(
    n_seeds          =("seed",           "size"),
    mean_rank_orig   =("rank_orig",      "mean"),
    mean_rank_full   =("rank_full",      "mean"),
    mean_rank_weather=("rank_no_weather","mean"),
    mean_rank_occasion=("rank_no_occasion","mean"),
    mean_rank_style  =("rank_no_style",  "mean"),
    hit10_orig       =("hit@10_orig",    "mean"),
    hit10_full       =("hit@10_full",    "mean"),
).reset_index()

# ── 6c) 嚴格過濾：full rank < orig rank ─────────────────
agg_combined = agg_combined[
    agg_combined["mean_rank_full"] < agg_combined["mean_rank_orig"]
].copy()

agg_combined["rank_gain"]      = agg_combined["mean_rank_orig"] - agg_combined["mean_rank_full"]
agg_combined["improved_hit10"] = (
    (agg_combined["hit10_orig"] < 0.5) & (agg_combined["hit10_full"] >= 0.5)
).astype(int)
agg_combined["category_name"]  = agg_combined["target_item_fg"].apply(map_category)

agg_combined = agg_combined.sort_values(
    ["improved_hit10", "rank_gain"], ascending=[False, False]
).reset_index(drop=True)

# ── 6d) 依「各因子移除後變差幅度」分別排序 ──────────────
# gain_weather  = no_weather rank - full rank（越大 = weather 移除後越差 = weather 貢獻越大）
agg_combined["gain_weather"]  = agg_combined["mean_rank_weather"]  - agg_combined["mean_rank_full"]
agg_combined["gain_occasion"] = agg_combined["mean_rank_occasion"] - agg_combined["mean_rank_full"]
agg_combined["gain_style"]    = agg_combined["mean_rank_style"]    - agg_combined["mean_rank_full"]

# 各因子驅動的 top cases，條件：
#   1. full 是五者中排名最好（mean_rank_full 最小）
#   2. 移除該因子後（no_x）是三個消融實驗中排名最差（mean_rank_no_x 最大）
#      → 代表該因子對這筆樣本貢獻最大

# full 是五者最好：已由 agg_combined 保證 full < orig；再加上 full < 三個 ablation
cond_full_best = (
    (agg_combined["mean_rank_full"] < agg_combined["mean_rank_weather"]) &
    (agg_combined["mean_rank_full"] < agg_combined["mean_rank_occasion"]) &
    (agg_combined["mean_rank_full"] < agg_combined["mean_rank_style"])
)

# weather 貢獻最大：no_weather 是三個 ablation 中最差
cond_weather_worst = (
    (agg_combined["mean_rank_weather"] >= agg_combined["mean_rank_occasion"]) &
    (agg_combined["mean_rank_weather"] >= agg_combined["mean_rank_style"])
)
# occasion 貢獻最大：no_occasion 是三個 ablation 中最差
cond_occasion_worst = (
    (agg_combined["mean_rank_occasion"] >= agg_combined["mean_rank_weather"]) &
    (agg_combined["mean_rank_occasion"] >= agg_combined["mean_rank_style"])
)
# style 貢獻最大：no_style 是三個 ablation 中最差
cond_style_worst = (
    (agg_combined["mean_rank_style"] >= agg_combined["mean_rank_weather"]) &
    (agg_combined["mean_rank_style"] >= agg_combined["mean_rank_occasion"])
)

agg_weather_driven = (
    agg_combined[cond_full_best & cond_weather_worst]
    .sort_values(["improved_hit10", "mean_rank_full"], ascending=[False, True])
    .reset_index(drop=True)
)
agg_occasion_driven = (
    agg_combined[cond_full_best & cond_occasion_worst]
    .sort_values(["improved_hit10", "mean_rank_full"], ascending=[False, True])
    .reset_index(drop=True)
)
agg_style_driven = (
    agg_combined[cond_full_best & cond_style_worst]
    .sort_values(["improved_hit10", "mean_rank_full"], ascending=[False, True])
    .reset_index(drop=True)
)

print(f"\n  Weather-driven  cases: {len(agg_weather_driven)}")
print(f"  Occasion-driven cases: {len(agg_occasion_driven)}")
print(f"  Style-driven    cases: {len(agg_style_driven)}")

display(agg_combined[[
    "set_id", "target_item_id", "category_name",
    "mean_rank_orig", "mean_rank_weather", "mean_rank_occasion",
    "mean_rank_style", "mean_rank_full", "rank_gain",
    "gain_weather", "gain_occasion", "gain_style"
]].head(10))


# =========================================================
# 7) 論文用個案展示（5 行：Orig / No-W / No-Occ / No-Style / Full）
# =========================================================
def show_full_ablation_case(agg_df, case_idx=0, topk=5, save_fig=False):
    """
    一張圖展示同一 case 的全部 5 個 variant：
      Row 0  Query Target header
      Row 1  Target image（置中）
      Row 2  Original (Baseline) header
      Row 3  Original top-k
      Row 4  No-Weather header
      Row 5  No-Weather top-k
      Row 6  No-Occasion header
      Row 7  No-Occasion top-k
      Row 8  No-Style header
      Row 9  No-Style top-k
      Row 10 Full (Proposed) header
      Row 11 Full top-k
    過濾條件：full rank < orig rank（已在 agg_combined 中保證）
    """
    if case_idx >= len(agg_df):
        print(f"case_idx={case_idx} out of range (total={len(agg_df)})")
        return

    case     = agg_df.iloc[case_idx]
    sid      = str(case["set_id"])
    tid      = str(case["target_item_id"])
    cat_name = case["category_name"]

    # 先取 mean rank（用於 header 顯示）
    mr_orig    = case["mean_rank_orig"]
    mr_weather = case["mean_rank_weather"]
    mr_occasion= case["mean_rank_occasion"]
    mr_style   = case["mean_rank_style"]
    mr_full    = case["mean_rank_full"]

    # 選代表 seed：rank_full 最接近 mean_rank_full 的那個
    # 這樣圖片位置和 header 顯示的 mean rank 最吻合
    cand = combined[
        (combined["set_id"].astype(str)         == sid) &
        (combined["target_item_id"].astype(str) == tid)
    ].copy()

    if cand.empty:
        print("No raw rows found.")
        return

    cand["_dist"] = (cand["rank_full"] - mr_full).abs()
    row = cand.sort_values("_dist").iloc[0]

    def _ids(col):
        v = row.get(col)
        return (v if isinstance(v, list) else [])[:topk]

    ids_orig     = _ids("top10_ids_list_orig")
    ids_weather  = _ids("top10_ids_list_no_weather")
    ids_occasion = _ids("top10_ids_list_no_occasion")
    ids_style    = _ids("top10_ids_list_no_style")
    ids_full     = _ids("top10_ids_list_full")

    # Header/圖片 rank：用代表 seed 的實際 rank（整數，與圖片完全對應）
    r_orig     = int(row["rank_orig"])
    r_weather  = int(row["rank_no_weather"])  if pd.notna(row.get("rank_no_weather"))  else None
    r_occasion = int(row["rank_no_occasion"]) if pd.notna(row.get("rank_no_occasion")) else None
    r_style    = int(row["rank_no_style"])    if pd.notna(row.get("rank_no_style"))    else None
    r_full     = int(row["rank_full"])

    # ── 文字摘要（同時顯示代表 seed rank 和 mean rank）────
    SEP  = "=" * 72
    DASH = "─" * 72
    print(f"\n{SEP}")
    print(f"  Case #{case_idx}  |  set={sid}  |  target={tid}  |  [{cat_name}]")
    print(DASH)
    print(f"  {'variant':>12s}  {'seed rank':>10s}  {'mean rank':>10s}")
    print(DASH)
    for lbl, rk, mrk in [
        ("orig",       r_orig,    mr_orig),
        ("no_weather", r_weather, mr_weather),
        ("no_occasion",r_occasion,mr_occasion),
        ("no_style",   r_style,   mr_style),
        ("full",       r_full,    mr_full),
    ]:
        if rk is None:
            print(f"  {lbl:>12s}  {'—':>10s}  {mrk:>10.1f}")
            continue
        arrow = "—" if lbl == "orig" else ("▲" if rk < r_orig else "▼")
        print(f"  {lbl:>12s}  {rk:>10d}  {mrk:>10.1f}  {arrow}")
    print(DASH)
    print(f"  [Original] {str(case.get('original_text',   ''))[:110]}")
    print(f"  [Full    ] {str(case.get('title_full_text', ''))[:110]}")
    print(f"  ↳ Weather : {case.get('weather_text','')}")
    print(f"  ↳ Occasion: {case.get('occasion_text','')}")
    print(f"  ↳ Style   : {case.get('style_text','')}")
    print(f"{SEP}\n")

    # ── result rows ──────────────────────────────────────
    result_rows = [
        (ids_orig,     r_orig,    "Original (Baseline)",    "#555555"),
        (ids_weather,  r_weather, "No-Weather (ablation)",  "#C0392B"),
        (ids_occasion, r_occasion,"No-Occasion (ablation)", "#E67E22"),
        (ids_style,    r_style,   "No-Style (ablation)",    "#8E44AD"),
        (ids_full,     r_full,    "Full (Proposed)",         "#2166AC"),
    ]
    n_result = len(result_rows)  # 5

    # ── GridSpec ─────────────────────────────────────────
    from matplotlib.gridspec import GridSpec
    HDR_H, IMG_H, TGT_H = 0.30, 2.5, 2.2
    height_ratios = [HDR_H, TGT_H] + [HDR_H, IMG_H] * n_result
    n_gs = 2 + n_result * 2

    fig_w = max(topk, 3) * 2.45 + 0.3
    fig_h = sum(height_ratios) + 0.5

    fig = plt.figure(figsize=(fig_w, fig_h))
    gs  = GridSpec(n_gs, topk, figure=fig,
                   height_ratios=height_ratios,
                   hspace=0.0, wspace=0.07,
                   top=0.97, bottom=0.03, left=0.02, right=0.98)

    def _draw(ax, img_id, is_hit=False, is_target=False):
        img = open_image_or_none(img_id)
        if img is not None:
            ax.imshow(img)
        else:
            ax.set_facecolor("#eeeeee")
            ax.text(0.5, 0.5, "N/A", ha="center", va="center",
                    fontsize=7, color="#aaaaaa", transform=ax.transAxes)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel(
            "✓ Hit" if is_hit else "",
            fontsize=8, color="#27AE60" if is_hit else "#666666",
            fontweight="bold" if is_hit else "normal", labelpad=2
        )
        if is_target: lw, ec = 4.0, "#111111"
        elif is_hit:  lw, ec = 5.5, "#27AE60"
        else:         lw, ec = 0.7, "#cccccc"
        for sp in ax.spines.values():
            sp.set_visible(True); sp.set_edgecolor(ec); sp.set_linewidth(lw)

    def _header(gs_row, text, color):
        ax_h = fig.add_subplot(gs[gs_row, :])
        ax_h.axis("off")
        ax_h.text(0.5, 0.45, text, ha="center", va="center",
                  fontsize=10, fontweight="bold", color=color,
                  transform=ax_h.transAxes)

    # Row 0: Target header
    _header(0, f"Query Target  —  {cat_name}", "#333333")
    # Row 1: Target image (centered)
    mid = topk // 2
    for c in range(topk):
        ax = fig.add_subplot(gs[1, c])
        if c == mid:
            _draw(ax, tid, is_target=True)
        else:
            ax.set_visible(False)

    # Row 2…: result rows
    for r, (ids, rank, label, color) in enumerate(result_rows):
        rank_str = str(rank) if rank is not None else "—"
        _header(2 + r * 2, f"{label}   (rank = {rank_str})", color)
        for i in range(topk):
            ax = fig.add_subplot(gs[2 + r * 2 + 1, i])
            if i < len(ids):
                iid = ids[i]
                _draw(ax, iid, is_hit=(str(iid) == str(tid)))
                ax.text(0.03, 0.97, f"#{i+1}", transform=ax.transAxes,
                        fontsize=7.5, va="top", ha="left", color="#444444",
                        bbox=dict(fc="white", ec="none", alpha=0.7, pad=1))
            else:
                ax.set_visible(False)

    fig.legend(
        handles=[mpatches.Patch(facecolor="#27AE60", edgecolor="#27AE60",
                                label="Ground-truth target retrieved")],
        loc="lower center", ncol=1, fontsize=8,
        frameon=False, bbox_to_anchor=(0.5, 0.0)
    )
    fig.suptitle(
        f"Case #{case_idx}  |  set={sid}  |  {cat_name}",
        fontsize=10, fontweight="bold", y=0.99
    )

    if SAVE_FIGS or save_fig:
        fname = f"full_ablation_case_idx{case_idx:02d}.pdf"
        fpath = os.path.join(FIG_DIR, fname)
        plt.savefig(fpath, dpi=300, bbox_inches="tight")
        print(f"[saved] {fpath}")
    plt.show()


# =========================================================
# 8) Category analysis
# =========================================================
def category_summary(df, left_name, right_name):
    grp = df.groupby("target_item_fg").agg(
        n          =("query_key",          "size"),
        left_hit10 =(f"hit@10_{left_name}",  "mean"),
        right_hit10=(f"hit@10_{right_name}", "mean"),
        left_rank  =(f"rank_{left_name}",    "median"),
        right_rank =(f"rank_{right_name}",   "median"),
    ).reset_index()
    grp["delta_hit10"] = grp["right_hit10"] - grp["left_hit10"]
    grp["delta_rank"]  = grp["right_rank"]  - grp["left_rank"]
    # 套用名稱
    grp["category_name"] = grp["target_item_fg"].apply(map_category)
    return grp.sort_values(
        ["delta_hit10", "n"], ascending=[False, False]
    ).reset_index(drop=True)


cat_orig_full = category_summary(m_orig_full, "orig",       "full")
cat_weather   = category_summary(m_weather,   "no_weather",  "full")
cat_occasion  = category_summary(m_occasion,  "no_occasion", "full")
cat_style     = category_summary(m_style,     "no_style",    "full")

print("\n=== Category analysis: Original → Proposed ===")
display(cat_orig_full[["category_name", "n", "left_hit10", "right_hit10", "delta_hit10"]].head(10))
print("\n=== Category analysis: Weather contribution ===")
display(cat_weather  [["category_name", "n", "left_hit10", "right_hit10", "delta_hit10"]].head(10))
print("\n=== Category analysis: Occasion contribution ===")
display(cat_occasion [["category_name", "n", "left_hit10", "right_hit10", "delta_hit10"]].head(10))
print("\n=== Category analysis: Style contribution ===")
display(cat_style    [["category_name", "n", "left_hit10", "right_hit10", "delta_hit10"]].head(10))


# =========================================================
# 9) 論文用 Category Bar Chart
# =========================================================
def plot_paper_category_bar(
    df, title,
    label_col="category_name", value_col="delta_hit10",
    top_n=8, save_name=None,
    color_pos="#2166AC", color_neg="#D6604D",
    figsize=(7, 4.2)
):
    plot_df = df.head(top_n).copy().sort_values(value_col, ascending=True)
    colors  = [color_pos if v >= 0 else color_neg for v in plot_df[value_col]]

    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.barh(
        plot_df[label_col].astype(str),
        plot_df[value_col],
        color=colors, edgecolor="white", height=0.6
    )

    # 數值標籤
    max_val = plot_df[value_col].abs().max()
    for bar, val in zip(bars, plot_df[value_col]):
        offset = max_val * 0.015 * (1 if val >= 0 else -1)
        ha     = "left" if val >= 0 else "right"
        ax.text(
            val + offset,
            bar.get_y() + bar.get_height() / 2,
            f"{val:+.3f}",
            va="center", ha=ha, fontsize=8.5
        )

    ax.axvline(0, color="gray", linestyle="--", linewidth=0.9)
    ax.set_xlabel("ΔHit@10  (Proposed − Baseline)", fontsize=10)
    ax.set_title(title, fontsize=11, fontweight="bold", pad=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="y", labelsize=9)
    plt.tight_layout()

    if SAVE_FIGS and save_name:
        fpath = os.path.join(FIG_DIR, save_name)
        plt.savefig(fpath, dpi=300, bbox_inches="tight")
        print(f"[saved] {fpath}")
    plt.show()


plot_paper_category_bar(
    cat_orig_full,
    title="Top Categories Improved by Proposed Description",
    top_n=8, save_name="fig_category_orig_vs_full.pdf"
)
plot_paper_category_bar(
    cat_weather,
    title="Categories Most Sensitive to Weather Information",
    top_n=8, save_name="fig_category_weather.pdf"
)
plot_paper_category_bar(
    cat_occasion,
    title="Categories Most Sensitive to Occasion Information",
    top_n=8, save_name="fig_category_occasion.pdf"
)
plot_paper_category_bar(
    cat_style,
    title="Categories Most Sensitive to Style Information",
    top_n=8, save_name="fig_category_style.pdf"
)


# =========================================================
# 10) Condition / Term analysis
# =========================================================
def explode_terms(df, terms_col):
    tmp = df.copy()
    tmp[terms_col] = tmp[terms_col].apply(
        lambda x: x if isinstance(x, list) else []
    )
    tmp = tmp.explode(terms_col)
    return tmp[
        tmp[terms_col].notna() &
        (tmp[terms_col].astype(str).str.strip() != "")
    ]


def term_summary(df, terms_col, left_name, right_name, min_n=20):
    tmp = explode_terms(df, terms_col)
    grp = tmp.groupby(terms_col).agg(
        n          =("query_key",          "size"),
        left_hit10 =(f"hit@10_{left_name}",  "mean"),
        right_hit10=(f"hit@10_{right_name}", "mean"),
        left_rank  =(f"rank_{left_name}",    "median"),
        right_rank =(f"rank_{right_name}",   "median"),
    ).reset_index()
    grp["delta_hit10"] = grp["right_hit10"] - grp["left_hit10"]
    grp["delta_rank"]  = grp["right_rank"]  - grp["left_rank"]
    return (
        grp[grp["n"] >= min_n]
        .sort_values(["delta_hit10", "n"], ascending=[False, False])
        .reset_index(drop=True)
    )


weather_terms_orig_full  = term_summary(m_orig_full, "weather_terms",  "orig",       "full")
occasion_terms_orig_full = term_summary(m_orig_full, "occasion_terms", "orig",       "full")
style_terms_orig_full    = term_summary(m_orig_full, "style_terms",    "orig",       "full")
weather_terms_factor     = term_summary(m_weather,   "weather_terms",  "no_weather",  "full")
occasion_terms_factor    = term_summary(m_occasion,  "occasion_terms", "no_occasion", "full")
style_terms_factor       = term_summary(m_style,     "style_terms",    "no_style",    "full")

print("\n=== Weather  terms (orig → full) ==="); display(weather_terms_orig_full.head(10))
print("\n=== Occasion terms (orig → full) ==="); display(occasion_terms_orig_full.head(10))
print("\n=== Style    terms (orig → full) ==="); display(style_terms_orig_full.head(10))
print("\n=== Weather  contribution ===");         display(weather_terms_factor.head(10))
print("\n=== Occasion contribution ===");         display(occasion_terms_factor.head(10))
print("\n=== Style    contribution ===");         display(style_terms_factor.head(10))


# Term bar charts（僅在有足夠資料時）
for term_df, col, title, fname in [
    (weather_terms_orig_full,  "weather_terms",
     "Top Weather Terms Improved by Proposed Description",
     "fig_weather_terms_orig_vs_full.pdf"),
    (occasion_terms_orig_full, "occasion_terms",
     "Top Occasion Terms Improved by Proposed Description",
     "fig_occasion_terms_orig_vs_full.pdf"),
    (style_terms_orig_full,    "style_terms",
     "Top Style Terms Improved by Proposed Description",
     "fig_style_terms_orig_vs_full.pdf"),
    (weather_terms_factor,     "weather_terms",
     "Weather Term Contribution (no_weather vs full)",
     "fig_weather_terms_factor.pdf"),
    (occasion_terms_factor,    "occasion_terms",
     "Occasion Term Contribution (no_occasion vs full)",
     "fig_occasion_terms_factor.pdf"),
    (style_terms_factor,       "style_terms",
     "Style Term Contribution (no_style vs full)",
     "fig_style_terms_factor.pdf"),
]:
    if len(term_df) > 0:
        plot_paper_category_bar(
            term_df, title=title,
            label_col=col, value_col="delta_hit10",
            top_n=8, save_name=fname,
            figsize=(7, 4)
        )


# =========================================================
# 11) Failure cases
# =========================================================
FAIL_GROUP_COLS = [
    "set_id", "target_item_id", "target_item_fg",
    "original_text", "title_full_text",
    "title_no_weather_text", "title_no_occasion_text", "title_no_style_text",
    "weather_text", "occasion_text", "style_text",
]


def aggregate_failure_candidates(df, left_name, right_name):
    out = df.copy()
    out["rank_loss"]   = out[f"rank_{right_name}"] - out[f"rank_{left_name}"]
    out["became_worse"] = (out[f"rank_{right_name}"] > out[f"rank_{left_name}"]).astype(int)

    existing = [c for c in FAIL_GROUP_COLS if c in out.columns]
    grp = out.groupby(existing, dropna=False).agg(
        n_seeds        =("seed",          "size"),
        mean_rank_left =(f"rank_{left_name}",  "mean"),
        mean_rank_right=(f"rank_{right_name}", "mean"),
        mean_rank_loss =("rank_loss",      "mean"),
        worse_count    =("became_worse",   "sum"),
    ).reset_index()

    grp["comparison"] = f"{left_name} -> {right_name}"
    grp["worse_rate"] = grp["worse_count"] / grp["n_seeds"]
    grp["category_name"] = grp["target_item_fg"].apply(map_category)

    return grp.sort_values(
        ["worse_rate", "mean_rank_loss"], ascending=[False, False]
    ).reset_index(drop=True)


fail_orig_full = aggregate_failure_candidates(m_orig_full, "orig",       "full")
fail_weather   = aggregate_failure_candidates(m_weather,   "no_weather",  "full")
fail_occasion  = aggregate_failure_candidates(m_occasion,  "no_occasion", "full")
fail_style     = aggregate_failure_candidates(m_style,     "no_style",    "full")

print("\n=== Failure cases: Original → Proposed ===")
display(fail_orig_full[["set_id","target_item_id","category_name",
                         "mean_rank_left","mean_rank_right","worse_rate"]].head(10))
print("\n=== Failure cases: Weather contribution ===")
display(fail_weather  [["set_id","target_item_id","category_name",
                         "mean_rank_left","mean_rank_right","worse_rate"]].head(10))
print("\n=== Failure cases: Occasion contribution ===")
display(fail_occasion [["set_id","target_item_id","category_name",
                         "mean_rank_left","mean_rank_right","worse_rate"]].head(10))
print("\n=== Failure cases: Style contribution ===")
display(fail_style    [["set_id","target_item_id","category_name",
                         "mean_rank_left","mean_rank_right","worse_rate"]].head(10))


# =========================================================
# 12) 儲存最終摘要表
# =========================================================
if SAVE_TABLES:
    pd.concat([
        agg_orig_full.assign(case_type="orig_to_full"),
        agg_weather  .assign(case_type="weather"),
        agg_occasion .assign(case_type="occasion"),
        agg_style    .assign(case_type="style"),
    ]).to_csv(os.path.join(TABLE_DIR, "qual_user_cases.csv"), index=False)

    pd.concat([
        fail_orig_full.assign(case_type="orig_to_full"),
        fail_weather  .assign(case_type="weather"),
        fail_occasion .assign(case_type="occasion"),
        fail_style    .assign(case_type="style"),
    ]).to_csv(os.path.join(TABLE_DIR, "qual_failure_cases.csv"), index=False)

    pd.concat([
        cat_orig_full.assign(case_type="orig_to_full"),
        cat_weather  .assign(case_type="weather"),
        cat_occasion .assign(case_type="occasion"),
        cat_style    .assign(case_type="style"),
    ]).to_csv(os.path.join(TABLE_DIR, "qual_category_summary.csv"), index=False)

    pd.concat([
        weather_terms_orig_full .assign(case_type="weather_orig_to_full"),
        occasion_terms_orig_full.assign(case_type="occasion_orig_to_full"),
        style_terms_orig_full   .assign(case_type="style_orig_to_full"),
        weather_terms_factor    .assign(case_type="weather_factor"),
        occasion_terms_factor   .assign(case_type="occasion_factor"),
        style_terms_factor      .assign(case_type="style_factor"),
    ]).to_csv(os.path.join(TABLE_DIR, "qual_condition_summary.csv"), index=False)

    print(f"[saved] tables → {TABLE_DIR}")


# =========================================================
# 13) 自動顯示各因子貢獻最大的 Top-N 個案
# =========================================================
TOP_N_CASES = 3   # 每個因子顯示幾個 case，可自行調整

FACTOR_GROUPS = [
    (agg_weather_driven,  "Weather  Factor  (no_weather  rank ≫ full rank)"),
    (agg_occasion_driven, "Occasion Factor  (no_occasion rank ≫ full rank)"),
    (agg_style_driven,    "Style    Factor  (no_style    rank ≫ full rank)"),
]

for agg_df, group_title in FACTOR_GROUPS:
    n_show = min(TOP_N_CASES, len(agg_df))
    print(f"\n{'#'*68}")
    print(f"  ▶  {group_title}")
    print(f"     有效案例數：{len(agg_df)}，顯示前 {n_show} 筆")
    print(f"{'#'*68}")
    for idx in range(n_show):
        show_full_ablation_case(agg_df, case_idx=idx, topk=TOPK)

print("\n✅ 全部個案展示完成")

[categories.csv] 載入 159 個類別，範例：{'2': 'undefined', '3': 'dress (all-body)', '4': 'dress (all-body)', '5': 'dress (all-body)'}
Loaded rows:
  original    : 17160
  full        : 17160
  no_weather  : 17160
  no_occasion : 17160
  no_style    : 17160
Loaded original titles : 68306


,set_id,original_text,original_title_only,original_url_part
0,219779031,spring trend colored denim Denim Jumpers,Denim Jumpers,spring trend colored denim
1,211158670,boot scootin outfit only 2500 Stacked-Heel Boots,Stacked-Heel Boots,boot scootin outfit only 2500
2,222049137,"mermaid for life ""Mermaid Hair, Don't Care""","""Mermaid Hair, Don't Care""",mermaid for life


Loaded fragments       : 35140


,set_id,title_full_text,weather_terms,occasion_terms,style_terms,weather_text,occasion_text,style_text,title_no_weather_text,title_no_occasion_text,title_no_style_text
0,201033839,"19°C, Effortless Mint & Lace Ensemble for a Br...","[19°C, Breezy Spring Day]",[],[Effortless Mint & Lace Ensemble],19°C | Breezy Spring Day,,Effortless Mint & Lace Ensemble,Effortless Mint & Lace Ensemble,"19°C, Effortless Mint & Lace Ensemble Breezy S...","19°C, Breezy Spring Day"
1,205002070,"24°C, Effortless Boho Vibes for Warm Days","[24°C, Warm Days]",[],[Effortless Boho Vibes],24°C | Warm Days,,Effortless Boho Vibes,Effortless Boho Vibes,"24°C, Effortless Boho Vibes Warm Days","24°C, Warm Days"
2,217778087,"17°C, Effortless Evening Glam with a Cozy Flar...",[17°C],[Evening],"[Effortless, Glam, Cozy, Flare Coat]",17°C,Evening,Effortless | Glam | Cozy | Flare Coat,Effortless Evening Glam Cozy Flare Coat,"17°C, Effortless Glam Cozy Flare Coat","17°C, Evening"


Merged rows:
  orig→full       : 17160
  weather→full    : 17160
  occasion→full   : 17160
  style→full      : 17160

  Weather-driven  cases: 62
  Occasion-driven cases: 99
  Style-driven    cases: 417


...


  Case #1  |  set=94771580  |  target=90931027  |  [dress (all-body)]
────────────────────────────────────────────────────────────────────────
       variant   seed rank   mean rank
────────────────────────────────────────────────────────────────────────
          orig          18       122.6  —
    no_weather           3         6.4  ▲
   no_occasion           8        24.4  ▲
      no_style          68        60.2  ▼
          full           5         3.0  ▲
────────────────────────────────────────────────────────────────────────
  [Original] salsa dance Salsa Lessons
  [Full    ] 23.6° C, Effortless Red Romance for Salsa Nights
  ↳ Weather : 23.6° C
  ↳ Occasion: Salsa Nights
  ↳ Style   : Effortless Red Romance



<Figure size 2008x2720 with 36 Axes>


  Case #2  |  set=198480847  |  target=171775539  |  [heels (shoes)]
────────────────────────────────────────────────────────────────────────
       variant   seed rank   mean rank
────────────────────────────────────────────────────────────────────────
          orig         222       159.8  —
    no_weather           3         4.8  ▲
   no_occasion           9         7.2  ▲
      no_style         101        81.0  ▲
          full           4         3.2  ▲
────────────────────────────────────────────────────────────────────────
  [Original] dinner cruise Cruise
  [Full    ] 27.2° C, Effortless coastal chic with beaded details and breezy sandals for a sun-drenched getaway.
  ↳ Weather : 27.2° C | sun-drenched
  ↳ Occasion: getaway
  ↳ Style   : Effortless coastal chic | beaded details | breezy sandals



<Figure size 2008x2720 with 36 Axes>


✅ 全部個案展示完成


In [30]:
import os
import re
import json
import glob
import ast
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from IPython.display import display

# =========================================================
# 0) CONFIG
# =========================================================
EXP_ROOT = "./experiments_none_only"
DETAIL_DIR = os.path.join(EXP_ROOT, "detail_cir")
if not os.path.exists(DETAIL_DIR):
    DETAIL_DIR = EXP_ROOT

DETAIL_RUN_TAG    = "none_subset"
DETAIL_SUBSET_TAG = "subset"

# 新描述 / fragments / ablation 來源
FRAG_JSONL = "/home/ester/valid_description/wos_split_results_v5_merged_retry_round3.jsonl"

# 原始描述來源
ORIGINAL_TITLE_JSON = "/home/ester/valid_description/polyvore_data/polyvore_outfits/polyvore_outfit_titles.json"

# Category mapping CSV
CATEGORY_CSV = "/home/ester/valid_description/polyvore_data/polyvore_outfits/categories.csv"

# 圖片路徑
IMAGE_ROOTS = [
    "/home/ester/valid_description/polyvore_data/polyvore_outfits/images",
]

SAVE_TABLES = False
SAVE_FIGS   = False      # True → 存 PDF/PNG

OUTPUT_DIR = "./qualitative_outputs_final"
TABLE_DIR  = os.path.join(OUTPUT_DIR, "tables")
FIG_DIR    = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TABLE_DIR,  exist_ok=True)
os.makedirs(FIG_DIR,    exist_ok=True)

VARIANTS = {
    "original"   : "outfitUrlTitle",
    "full"       : "NewoutfitUrlTitle",
    "no_weather" : "no_weather",
    "no_occasion": "no_occasion",
    "no_style"   : "no_style",
}

TOPK = 5

plt.rcParams.update({
    "figure.dpi"         : 160,
    "font.size"          : 10,
    "axes.spines.top"    : False,
    "axes.spines.right"  : False,
    "font.family"        : "DejaVu Sans",
})


# =========================================================
# 0b) Category mapping
# =========================================================
def load_category_map(csv_path):
    """
    讀取 categories.csv。
    支援格式：col0=id, col1=name, col2=group（無 header）
    回傳 dict: {str(id): "name (group)"}  例如 "dress (all-body)"
    若無 group 欄則僅回傳 name。
    同一 id 可能對應多個 name（取第一筆）。
    """
    if not os.path.exists(csv_path):
        print(f"[WARNING] 找不到 categories.csv：{csv_path}")
        print("         → category 欄位將顯示原始數字 ID")
        return {}

    # 先嘗試無 header 讀取，若第一欄為數字即為無 header 格式
    df_raw = pd.read_csv(csv_path, header=None)
    first_val = str(df_raw.iloc[0, 0]).strip()
    if first_val.isdigit():
        df = df_raw.copy()
    else:
        # 有 header：自動偵測欄位
        df = pd.read_csv(csv_path)
        id_candidates   = ["id", "category_id", "fg"]
        name_candidates = ["name", "category_name", "category"]
        grp_candidates  = ["group", "supercategory", "type", "parent"]
        id_col  = next((c for c in id_candidates   if c in df.columns), df.columns[0])
        nm_col  = next((c for c in name_candidates if c in df.columns), df.columns[1])
        grp_col = next((c for c in grp_candidates  if c in df.columns), None)
        df = df.rename(columns={id_col: 0, nm_col: 1})
        if grp_col:
            df = df.rename(columns={grp_col: 2})

    has_group = df.shape[1] >= 3

    result = {}
    for _, row in df.iterrows():
        cid = str(row[0]).strip()
        if cid in result:          # 同 id 取第一筆
            continue
        name = str(row[1]).strip() if pd.notna(row[1]) else cid
        if has_group and pd.notna(row[2]):
            grp   = str(row[2]).strip()
            label = f"{name} ({grp})"
        else:
            label = name
        result[cid] = label

    sample = dict(list(result.items())[:4])
    print(f"[categories.csv] 載入 {len(result)} 個類別，範例：{sample}")
    return result


CATEGORY_NAME_MAP = load_category_map(CATEGORY_CSV)


def map_category(x):
    """數字 → 類別名稱；找不到就直接回傳原始字串。"""
    return CATEGORY_NAME_MAP.get(str(x), str(x))


# =========================================================
# 1) 讀取 detail csv
# =========================================================
def load_one_variant_detail(variant_name):
    pattern = os.path.join(
        DETAIL_DIR,
        f"detail_cir_{variant_name}_{DETAIL_SUBSET_TAG}_{DETAIL_RUN_TAG}_seed*.csv"
    )
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"找不到 detail 檔案：{pattern}")

    dfs = []
    for fp in files:
        df = pd.read_csv(fp)
        df["variant"]     = variant_name
        df["source_file"] = os.path.basename(fp)
        dfs.append(df)

    return pd.concat(dfs, axis=0, ignore_index=True)


detail_original   = load_one_variant_detail(VARIANTS["original"])
detail_full       = load_one_variant_detail(VARIANTS["full"])
detail_no_weather = load_one_variant_detail(VARIANTS["no_weather"])
detail_no_occasion= load_one_variant_detail(VARIANTS["no_occasion"])
detail_no_style   = load_one_variant_detail(VARIANTS["no_style"])

print("Loaded rows:")
for name, df in [("original", detail_original), ("full", detail_full),
                 ("no_weather", detail_no_weather), ("no_occasion", detail_no_occasion),
                 ("no_style", detail_no_style)]:
    print(f"  {name:12s}: {len(df)}")


# =========================================================
# 2) Schema 正規化
# =========================================================
def normalize_detail_df(df):
    out = df.copy()
    required = ["seed", "set_id", "target_item_id", "target_item_fg",
                "rank", "hit@10", "top10_ids"]
    for c in required:
        if c not in out.columns:
            raise KeyError(f"detail csv 缺少必要欄位：{c}")

    out["seed"]           = out["seed"].astype(int)
    out["set_id"]         = out["set_id"].astype(str)
    out["target_item_id"] = out["target_item_id"].astype(str)
    out["target_item_fg"] = out["target_item_fg"].astype(str)
    out["rank"]           = pd.to_numeric(out["rank"],    errors="coerce")
    out["hit@10"]         = pd.to_numeric(out["hit@10"],  errors="coerce")

    def parse_top10(x):
        if pd.isna(x):
            return []
        if isinstance(x, list):
            return [str(i) for i in x]
        try:
            return [str(i) for i in json.loads(x)]
        except Exception:
            try:
                v = ast.literal_eval(x)
                return [str(i) for i in v] if isinstance(v, list) else []
            except Exception:
                return []

    out["top10_ids_list"] = out["top10_ids"].apply(parse_top10)
    out["query_key"] = (
        out["seed"].astype(str) + "||" +
        out["set_id"]           + "||" +
        out["target_item_id"]
    )
    return out


detail_original    = normalize_detail_df(detail_original)
detail_full        = normalize_detail_df(detail_full)
detail_no_weather  = normalize_detail_df(detail_no_weather)
detail_no_occasion = normalize_detail_df(detail_no_occasion)
detail_no_style    = normalize_detail_df(detail_no_style)


# =========================================================
# 3) 載入 metadata
# =========================================================
def normalize_text(x):
    if x is None:
        return ""
    if isinstance(x, str):
        return x.strip()
    if isinstance(x, list):
        return " | ".join(str(i).strip() for i in x if str(i).strip())
    return str(x).strip()


def load_original_titles_json(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"找不到原始描述檔案：{path}")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []
    for sid, value in data.items():
        original_text = original_title_only = original_url_part = ""
        if isinstance(value, str):
            original_text = value.strip()
        elif isinstance(value, dict):
            title_part = normalize_text(value.get("title", ""))
            url_part   = normalize_text(value.get("url_name", value.get("url", "")))
            original_text      = " ".join(p for p in [url_part, title_part] if p).strip()
            original_title_only = title_part
            original_url_part   = url_part
        else:
            original_text = normalize_text(value)

        rows.append({
            "set_id"            : str(sid),
            "original_text"     : original_text,
            "original_title_only": original_title_only,
            "original_url_part" : original_url_part,
        })
    return pd.DataFrame(rows)


def load_fragments_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"找不到 fragments 檔案：{path}")
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if obj.get("valid") is not True:
                continue

            frags          = obj.get("fragments", {})
            title_ablation = obj.get("title_ablation", {})
            weather_terms  = frags.get("weather",  [])
            occasion_terms = frags.get("occasion", [])
            style_terms    = frags.get("style",    [])

            rows.append({
                "set_id"               : str(obj.get("id", "")),
                "title_full_text"      : normalize_text(obj.get("title", "")),
                "weather_terms"        : weather_terms  if isinstance(weather_terms,  list) else [],
                "occasion_terms"       : occasion_terms if isinstance(occasion_terms, list) else [],
                "style_terms"          : style_terms    if isinstance(style_terms,    list) else [],
                "weather_text"         : normalize_text(weather_terms),
                "occasion_text"        : normalize_text(occasion_terms),
                "style_text"           : normalize_text(style_terms),
                "title_no_weather_text" : normalize_text(title_ablation.get("no_weather",  "")),
                "title_no_occasion_text": normalize_text(title_ablation.get("no_occasion", "")),
                "title_no_style_text"   : normalize_text(title_ablation.get("no_style",    "")),
            })
    return pd.DataFrame(rows)


orig_title_df = load_original_titles_json(ORIGINAL_TITLE_JSON)
frag_df       = load_fragments_jsonl(FRAG_JSONL)

print(f"Loaded original titles : {len(orig_title_df)}")
display(orig_title_df.head(3))
print(f"Loaded fragments       : {len(frag_df)}")
display(frag_df.head(3))


# =========================================================
# 4) Merge comparison pairs
# =========================================================
def merge_pair(left_df, right_df, left_name, right_name):
    left = left_df[[
        "query_key", "seed", "set_id", "target_item_id", "target_item_fg",
        "rank", "hit@10", "top10_ids", "top10_ids_list"
    ]].copy()
    right = right_df[[
        "query_key", "rank", "hit@10", "top10_ids", "top10_ids_list"
    ]].copy()

    merged = (
        left
        .merge(right, on="query_key", how="inner",
               suffixes=(f"_{left_name}", f"_{right_name}"))
        .merge(orig_title_df, on="set_id", how="left")
        .merge(frag_df,       on="set_id", how="left")
    )
    return merged


m_orig_full = merge_pair(detail_original,    detail_full, "orig",       "full")
m_weather   = merge_pair(detail_no_weather,  detail_full, "no_weather",  "full")
m_occasion  = merge_pair(detail_no_occasion, detail_full, "no_occasion", "full")
m_style     = merge_pair(detail_no_style,    detail_full, "no_style",    "full")

print("Merged rows:")
for name, df in [("orig→full", m_orig_full), ("weather→full", m_weather),
                 ("occasion→full", m_occasion), ("style→full", m_style)]:
    print(f"  {name:16s}: {len(df)}")


# =========================================================
# 5) 圖片工具
# =========================================================
def find_image_by_id(img_id):
    for root in IMAGE_ROOTS:
        for ext in (".jpg", ".jpeg", ".png", ".webp"):
            p = Path(root) / f"{img_id}{ext}"
            if p.exists():
                return str(p)
    return None


def open_image_or_none(img_id):
    fp = find_image_by_id(str(img_id))
    if fp is None:
        return None
    try:
        return Image.open(fp).convert("RGB")
    except Exception:
        return None


# =========================================================
# 6) Combined aggregation（5 variants 同一 case）
# =========================================================

# 保留舊的 aggregate_case_candidates 供 category/term analysis 使用
GROUP_COLS = [
    "set_id", "target_item_id", "target_item_fg",
    "original_text", "original_title_only", "original_url_part",
    "title_full_text",
    "title_no_weather_text", "title_no_occasion_text", "title_no_style_text",
    "weather_text", "occasion_text", "style_text",
]

def aggregate_case_candidates(df, left_name, right_name):
    out = df.copy()
    out["rank_gain"]      = out[f"rank_{left_name}"] - out[f"rank_{right_name}"]
    out["improved_hit10"] = (
        (out[f"rank_{left_name}"] > 10) & (out[f"rank_{right_name}"] <= 10)
    ).astype(int)
    out["worse"] = (out[f"rank_{right_name}"] > out[f"rank_{left_name}"]).astype(int)
    existing = [c for c in GROUP_COLS if c in out.columns]
    grp = out.groupby(existing, dropna=False).agg(
        n_seeds        =("seed",               "size"),
        mean_rank_left =(f"rank_{left_name}",  "mean"),
        mean_rank_right=(f"rank_{right_name}", "mean"),
        mean_rank_gain =("rank_gain",          "mean"),
        success_shift  =("improved_hit10",     "sum"),
        worse_count    =("worse",              "sum"),
        win_count      =("rank_gain",          lambda s: (s > 0).sum()),
    ).reset_index()
    grp["comparison"] = f"{left_name} -> {right_name}"
    grp["win_rate"]   = grp["win_count"] / grp["n_seeds"]
    return grp.sort_values(
        ["success_shift", "win_rate", "mean_rank_gain"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

agg_orig_full = aggregate_case_candidates(m_orig_full, "orig",       "full")
agg_weather   = aggregate_case_candidates(m_weather,   "no_weather",  "full")
agg_occasion  = aggregate_case_candidates(m_occasion,  "no_occasion", "full")
agg_style     = aggregate_case_candidates(m_style,     "no_style",    "full")

# ── 6a) 建立 combined dataframe（join 全部 4 個 merged df）──
def _slim(df, variant):
    return df[["query_key", f"rank_{variant}", f"top10_ids_list_{variant}"]].copy()

combined = (
    m_orig_full[[
        "query_key", "seed", "set_id", "target_item_id", "target_item_fg",
        "original_text", "title_full_text",
        "title_no_weather_text", "title_no_occasion_text", "title_no_style_text",
        "weather_text", "occasion_text", "style_text",
        "rank_orig", "rank_full",
        "top10_ids_list_orig", "top10_ids_list_full",
        "hit@10_orig", "hit@10_full",
    ]]
    .merge(_slim(m_weather,  "no_weather"),  on="query_key", how="left")
    .merge(_slim(m_occasion, "no_occasion"), on="query_key", how="left")
    .merge(_slim(m_style,    "no_style"),    on="query_key", how="left")
)

# ── 6b) 聚合：跨 seed 取平均 rank ────────────────────────
COMB_GROUP_COLS = [
    "set_id", "target_item_id", "target_item_fg",
    "original_text", "title_full_text",
    "title_no_weather_text", "title_no_occasion_text", "title_no_style_text",
    "weather_text", "occasion_text", "style_text",
]
existing_comb = [c for c in COMB_GROUP_COLS if c in combined.columns]

agg_combined = combined.groupby(existing_comb, dropna=False).agg(
    n_seeds          =("seed",           "size"),
    mean_rank_orig   =("rank_orig",      "mean"),
    mean_rank_full   =("rank_full",      "mean"),
    mean_rank_weather=("rank_no_weather","mean"),
    mean_rank_occasion=("rank_no_occasion","mean"),
    mean_rank_style  =("rank_no_style",  "mean"),
    hit10_orig       =("hit@10_orig",    "mean"),
    hit10_full       =("hit@10_full",    "mean"),
).reset_index()

# ── 6c) 嚴格過濾：full rank < orig rank ─────────────────
agg_combined = agg_combined[
    agg_combined["mean_rank_full"] < agg_combined["mean_rank_orig"]
].copy()

agg_combined["rank_gain"]      = agg_combined["mean_rank_orig"] - agg_combined["mean_rank_full"]
agg_combined["improved_hit10"] = (
    (agg_combined["hit10_orig"] < 0.5) & (agg_combined["hit10_full"] >= 0.5)
).astype(int)
agg_combined["category_name"]  = agg_combined["target_item_fg"].apply(map_category)

agg_combined = agg_combined.sort_values(
    ["improved_hit10", "rank_gain"], ascending=[False, False]
).reset_index(drop=True)

# ── 6d) 依「各因子移除後變差幅度」分別排序 ──────────────
# gain_weather  = no_weather rank - full rank（越大 = weather 移除後越差 = weather 貢獻越大）
agg_combined["gain_weather"]  = agg_combined["mean_rank_weather"]  - agg_combined["mean_rank_full"]
agg_combined["gain_occasion"] = agg_combined["mean_rank_occasion"] - agg_combined["mean_rank_full"]
agg_combined["gain_style"]    = agg_combined["mean_rank_style"]    - agg_combined["mean_rank_full"]

# 各因子驅動的 top cases，條件：
#   1. full 是五者中排名最好（mean_rank_full 最小）
#   2. 移除該因子後（no_x）是三個消融實驗中排名最差（mean_rank_no_x 最大）
#      → 代表該因子對這筆樣本貢獻最大

# full 是五者最好：已由 agg_combined 保證 full < orig；再加上 full < 三個 ablation
cond_full_best = (
    (agg_combined["mean_rank_full"] < agg_combined["mean_rank_weather"]) &
    (agg_combined["mean_rank_full"] < agg_combined["mean_rank_occasion"]) &
    (agg_combined["mean_rank_full"] < agg_combined["mean_rank_style"])
)

# weather 貢獻最大：no_weather 是三個 ablation 中最差
cond_weather_worst = (
    (agg_combined["mean_rank_weather"] >= agg_combined["mean_rank_occasion"]) &
    (agg_combined["mean_rank_weather"] >= agg_combined["mean_rank_style"])
)
# occasion 貢獻最大：no_occasion 是三個 ablation 中最差
cond_occasion_worst = (
    (agg_combined["mean_rank_occasion"] >= agg_combined["mean_rank_weather"]) &
    (agg_combined["mean_rank_occasion"] >= agg_combined["mean_rank_style"])
)
# style 貢獻最大：no_style 是三個 ablation 中最差
cond_style_worst = (
    (agg_combined["mean_rank_style"] >= agg_combined["mean_rank_weather"]) &
    (agg_combined["mean_rank_style"] >= agg_combined["mean_rank_occasion"])
)

agg_weather_driven = (
    agg_combined[cond_full_best & cond_weather_worst]
    .sort_values(["improved_hit10", "mean_rank_full"], ascending=[False, True])
    .reset_index(drop=True)
)
agg_occasion_driven = (
    agg_combined[cond_full_best & cond_occasion_worst]
    .sort_values(["improved_hit10", "mean_rank_full"], ascending=[False, True])
    .reset_index(drop=True)
)
agg_style_driven = (
    agg_combined[cond_full_best & cond_style_worst]
    .sort_values(["improved_hit10", "mean_rank_full"], ascending=[False, True])
    .reset_index(drop=True)
)

print(f"\n  Weather-driven  cases: {len(agg_weather_driven)}")
print(f"  Occasion-driven cases: {len(agg_occasion_driven)}")
print(f"  Style-driven    cases: {len(agg_style_driven)}")

display(agg_combined[[
    "set_id", "target_item_id", "category_name",
    "mean_rank_orig", "mean_rank_weather", "mean_rank_occasion",
    "mean_rank_style", "mean_rank_full", "rank_gain",
    "gain_weather", "gain_occasion", "gain_style"
]].head(10))


# =========================================================
# 7) 論文用個案展示（5 行：Orig / No-W / No-Occ / No-Style / Full）
# =========================================================
def show_full_ablation_case(agg_df, case_idx=0, topk=5, save_fig=False):
    """
    一張圖展示同一 case 的全部 5 個 variant：
      Row 0  Query Target header
      Row 1  Target image（置中）
      Row 2  Original (Baseline) header
      Row 3  Original top-k
      Row 4  No-Weather header
      Row 5  No-Weather top-k
      Row 6  No-Occasion header
      Row 7  No-Occasion top-k
      Row 8  No-Style header
      Row 9  No-Style top-k
      Row 10 Full (Proposed) header
      Row 11 Full top-k
    過濾條件：full rank < orig rank（已在 agg_combined 中保證）
    """
    if case_idx >= len(agg_df):
        print(f"case_idx={case_idx} out of range (total={len(agg_df)})")
        return

    case     = agg_df.iloc[case_idx]
    sid      = str(case["set_id"])
    tid      = str(case["target_item_id"])
    cat_name = case["category_name"]

    # 先取 mean rank（用於 header 顯示）
    mr_orig    = case["mean_rank_orig"]
    mr_weather = case["mean_rank_weather"]
    mr_occasion= case["mean_rank_occasion"]
    mr_style   = case["mean_rank_style"]
    mr_full    = case["mean_rank_full"]

    # 選代表 seed：rank_full 最接近 mean_rank_full 的那個
    # 這樣圖片位置和 header 顯示的 mean rank 最吻合
    cand = combined[
        (combined["set_id"].astype(str)         == sid) &
        (combined["target_item_id"].astype(str) == tid)
    ].copy()

    if cand.empty:
        print("No raw rows found.")
        return

    cand["_dist"] = (cand["rank_full"] - mr_full).abs()
    row = cand.sort_values("_dist").iloc[0]

    def _ids(col):
        v = row.get(col)
        return (v if isinstance(v, list) else [])[:topk]

    ids_orig     = _ids("top10_ids_list_orig")
    ids_weather  = _ids("top10_ids_list_no_weather")
    ids_occasion = _ids("top10_ids_list_no_occasion")
    ids_style    = _ids("top10_ids_list_no_style")
    ids_full     = _ids("top10_ids_list_full")

    # Header/圖片 rank：用代表 seed 的實際 rank（整數，與圖片完全對應）
    r_orig     = int(row["rank_orig"])
    r_weather  = int(row["rank_no_weather"])  if pd.notna(row.get("rank_no_weather"))  else None
    r_occasion = int(row["rank_no_occasion"]) if pd.notna(row.get("rank_no_occasion")) else None
    r_style    = int(row["rank_no_style"])    if pd.notna(row.get("rank_no_style"))    else None
    r_full     = int(row["rank_full"])

    # ── 文字摘要（同時顯示代表 seed rank 和 mean rank）────
    SEP  = "=" * 72
    DASH = "─" * 72
    print(f"\n{SEP}")
    print(f"  Case #{case_idx}  |  set={sid}  |  target={tid}  |  [{cat_name}]")
    print(DASH)
    print(f"  {'variant':>12s}  {'seed rank':>10s}  {'mean rank':>10s}")
    print(DASH)
    for lbl, rk, mrk in [
        ("orig",       r_orig,    mr_orig),
        ("no_weather", r_weather, mr_weather),
        ("no_occasion",r_occasion,mr_occasion),
        ("no_style",   r_style,   mr_style),
        ("full",       r_full,    mr_full),
    ]:
        if rk is None:
            print(f"  {lbl:>12s}  {'—':>10s}  {mrk:>10.1f}")
            continue
        arrow = "—" if lbl == "orig" else ("▲" if rk < r_orig else "▼")
        print(f"  {lbl:>12s}  {rk:>10d}  {mrk:>10.1f}  {arrow}")
    print(DASH)
    print(f"  [Original] {str(case.get('original_text',   ''))[:110]}")
    print(f"  [Full    ] {str(case.get('title_full_text', ''))[:110]}")
    print(f"  ↳ Weather : {case.get('weather_text','')}")
    print(f"  ↳ Occasion: {case.get('occasion_text','')}")
    print(f"  ↳ Style   : {case.get('style_text','')}")
    print(f"{SEP}\n")

    # ── result rows ──────────────────────────────────────
    result_rows = [
        (ids_orig,     r_orig,    "Original (Baseline)",    "#555555"),
        (ids_weather,  r_weather, "No-Weather (ablation)",  "#C0392B"),
        (ids_occasion, r_occasion,"No-Occasion (ablation)", "#E67E22"),
        (ids_style,    r_style,   "No-Style (ablation)",    "#8E44AD"),
        (ids_full,     r_full,    "Full (Proposed)",         "#2166AC"),
    ]
    n_result = len(result_rows)  # 5

    # ── GridSpec ─────────────────────────────────────────
    from matplotlib.gridspec import GridSpec
    HDR_H, IMG_H, TGT_H = 0.30, 2.5, 2.2
    height_ratios = [HDR_H, TGT_H] + [HDR_H, IMG_H] * n_result
    n_gs = 2 + n_result * 2

    fig_w = max(topk, 3) * 2.45 + 0.3
    fig_h = sum(height_ratios) + 0.5

    fig = plt.figure(figsize=(fig_w, fig_h))
    gs  = GridSpec(n_gs, topk, figure=fig,
                   height_ratios=height_ratios,
                   hspace=0.0, wspace=0.07,
                   top=0.97, bottom=0.03, left=0.02, right=0.98)

    def _draw(ax, img_id, is_hit=False, is_target=False):
        img = open_image_or_none(img_id)
        if img is not None:
            ax.imshow(img)
        else:
            ax.set_facecolor("#eeeeee")
            ax.text(0.5, 0.5, "N/A", ha="center", va="center",
                    fontsize=7, color="#aaaaaa", transform=ax.transAxes)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel(
            "✓ Hit" if is_hit else "",
            fontsize=8, color="#27AE60" if is_hit else "#666666",
            fontweight="bold" if is_hit else "normal", labelpad=2
        )
        if is_target: lw, ec = 4.0, "#111111"
        elif is_hit:  lw, ec = 5.5, "#27AE60"
        else:         lw, ec = 0.7, "#cccccc"
        for sp in ax.spines.values():
            sp.set_visible(True); sp.set_edgecolor(ec); sp.set_linewidth(lw)

    def _header(gs_row, text, color):
        ax_h = fig.add_subplot(gs[gs_row, :])
        ax_h.axis("off")
        ax_h.text(0.5, 0.45, text, ha="center", va="center",
                  fontsize=10, fontweight="bold", color=color,
                  transform=ax_h.transAxes)

    # Row 0: Target header
    _header(0, f"Query Target  —  {cat_name}", "#333333")
    # Row 1: Target image (centered)
    mid = topk // 2
    for c in range(topk):
        ax = fig.add_subplot(gs[1, c])
        if c == mid:
            _draw(ax, tid, is_target=True)
        else:
            ax.set_visible(False)

    # Row 2…: result rows
    for r, (ids, rank, label, color) in enumerate(result_rows):
        rank_str = str(rank) if rank is not None else "—"
        _header(2 + r * 2, f"{label}   (rank = {rank_str})", color)
        for i in range(topk):
            ax = fig.add_subplot(gs[2 + r * 2 + 1, i])
            if i < len(ids):
                iid = ids[i]
                _draw(ax, iid, is_hit=(str(iid) == str(tid)))
                ax.text(0.03, 0.97, f"#{i+1}", transform=ax.transAxes,
                        fontsize=7.5, va="top", ha="left", color="#444444",
                        bbox=dict(fc="white", ec="none", alpha=0.7, pad=1))
            else:
                ax.set_visible(False)

    fig.legend(
        handles=[mpatches.Patch(facecolor="#27AE60", edgecolor="#27AE60",
                                label="Ground-truth target retrieved")],
        loc="lower center", ncol=1, fontsize=8,
        frameon=False, bbox_to_anchor=(0.5, 0.0)
    )
    fig.suptitle(
        f"Case #{case_idx}  |  set={sid}  |  {cat_name}",
        fontsize=10, fontweight="bold", y=0.99
    )

    if SAVE_FIGS or save_fig:
        fname = f"full_ablation_case_idx{case_idx:02d}.pdf"
        fpath = os.path.join(FIG_DIR, fname)
        plt.savefig(fpath, dpi=300, bbox_inches="tight")
        print(f"[saved] {fpath}")
    plt.show()


# =========================================================
# 8) Category analysis
# =========================================================
def category_summary(df, left_name, right_name):
    grp = df.groupby("target_item_fg").agg(
        n          =("query_key",          "size"),
        left_hit10 =(f"hit@10_{left_name}",  "mean"),
        right_hit10=(f"hit@10_{right_name}", "mean"),
        left_rank  =(f"rank_{left_name}",    "median"),
        right_rank =(f"rank_{right_name}",   "median"),
    ).reset_index()
    grp["delta_hit10"] = grp["right_hit10"] - grp["left_hit10"]
    grp["delta_rank"]  = grp["right_rank"]  - grp["left_rank"]
    # 套用名稱
    grp["category_name"] = grp["target_item_fg"].apply(map_category)
    return grp.sort_values(
        ["delta_hit10", "n"], ascending=[False, False]
    ).reset_index(drop=True)


cat_orig_full = category_summary(m_orig_full, "orig",       "full")
cat_weather   = category_summary(m_weather,   "no_weather",  "full")
cat_occasion  = category_summary(m_occasion,  "no_occasion", "full")
cat_style     = category_summary(m_style,     "no_style",    "full")

print("\n=== Category analysis: Original → Proposed ===")
display(cat_orig_full[["category_name", "n", "left_hit10", "right_hit10", "delta_hit10"]].head(10))
print("\n=== Category analysis: Weather contribution ===")
display(cat_weather  [["category_name", "n", "left_hit10", "right_hit10", "delta_hit10"]].head(10))
print("\n=== Category analysis: Occasion contribution ===")
display(cat_occasion [["category_name", "n", "left_hit10", "right_hit10", "delta_hit10"]].head(10))
print("\n=== Category analysis: Style contribution ===")
display(cat_style    [["category_name", "n", "left_hit10", "right_hit10", "delta_hit10"]].head(10))


# =========================================================
# 9) 論文用 Category Bar Chart
# =========================================================
def plot_paper_category_bar(
    df, title,
    label_col="category_name", value_col="delta_hit10",
    top_n=8, save_name=None,
    color_pos="#2166AC", color_neg="#D6604D",
    figsize=(7, 4.2)
):
    plot_df = df.head(top_n).copy().sort_values(value_col, ascending=True)
    colors  = [color_pos if v >= 0 else color_neg for v in plot_df[value_col]]

    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.barh(
        plot_df[label_col].astype(str),
        plot_df[value_col],
        color=colors, edgecolor="white", height=0.6
    )

    # 數值標籤
    max_val = plot_df[value_col].abs().max()
    for bar, val in zip(bars, plot_df[value_col]):
        offset = max_val * 0.015 * (1 if val >= 0 else -1)
        ha     = "left" if val >= 0 else "right"
        ax.text(
            val + offset,
            bar.get_y() + bar.get_height() / 2,
            f"{val:+.3f}",
            va="center", ha=ha, fontsize=8.5
        )

    ax.axvline(0, color="gray", linestyle="--", linewidth=0.9)
    ax.set_xlabel("ΔHit@10  (Proposed − Baseline)", fontsize=10)
    ax.set_title(title, fontsize=11, fontweight="bold", pad=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="y", labelsize=9)
    plt.tight_layout()

    if SAVE_FIGS and save_name:
        fpath = os.path.join(FIG_DIR, save_name)
        plt.savefig(fpath, dpi=300, bbox_inches="tight")
        print(f"[saved] {fpath}")
    plt.show()


plot_paper_category_bar(
    cat_orig_full,
    title="Top Categories Improved by Proposed Description",
    top_n=8, save_name="fig_category_orig_vs_full.pdf"
)
plot_paper_category_bar(
    cat_weather,
    title="Categories Most Sensitive to Weather Information",
    top_n=8, save_name="fig_category_weather.pdf"
)
plot_paper_category_bar(
    cat_occasion,
    title="Categories Most Sensitive to Occasion Information",
    top_n=8, save_name="fig_category_occasion.pdf"
)
plot_paper_category_bar(
    cat_style,
    title="Categories Most Sensitive to Style Information",
    top_n=8, save_name="fig_category_style.pdf"
)


# =========================================================
# 10) Condition / Term analysis
# =========================================================
def explode_terms(df, terms_col):
    tmp = df.copy()
    tmp[terms_col] = tmp[terms_col].apply(
        lambda x: x if isinstance(x, list) else []
    )
    tmp = tmp.explode(terms_col)
    return tmp[
        tmp[terms_col].notna() &
        (tmp[terms_col].astype(str).str.strip() != "")
    ]


def term_summary(df, terms_col, left_name, right_name, min_n=20):
    tmp = explode_terms(df, terms_col)
    grp = tmp.groupby(terms_col).agg(
        n          =("query_key",          "size"),
        left_hit10 =(f"hit@10_{left_name}",  "mean"),
        right_hit10=(f"hit@10_{right_name}", "mean"),
        left_rank  =(f"rank_{left_name}",    "median"),
        right_rank =(f"rank_{right_name}",   "median"),
    ).reset_index()
    grp["delta_hit10"] = grp["right_hit10"] - grp["left_hit10"]
    grp["delta_rank"]  = grp["right_rank"]  - grp["left_rank"]
    return (
        grp[grp["n"] >= min_n]
        .sort_values(["delta_hit10", "n"], ascending=[False, False])
        .reset_index(drop=True)
    )


weather_terms_orig_full  = term_summary(m_orig_full, "weather_terms",  "orig",       "full")
occasion_terms_orig_full = term_summary(m_orig_full, "occasion_terms", "orig",       "full")
style_terms_orig_full    = term_summary(m_orig_full, "style_terms",    "orig",       "full")
weather_terms_factor     = term_summary(m_weather,   "weather_terms",  "no_weather",  "full")
occasion_terms_factor    = term_summary(m_occasion,  "occasion_terms", "no_occasion", "full")
style_terms_factor       = term_summary(m_style,     "style_terms",    "no_style",    "full")

print("\n=== Weather  terms (orig → full) ==="); display(weather_terms_orig_full.head(10))
print("\n=== Occasion terms (orig → full) ==="); display(occasion_terms_orig_full.head(10))
print("\n=== Style    terms (orig → full) ==="); display(style_terms_orig_full.head(10))
print("\n=== Weather  contribution ===");         display(weather_terms_factor.head(10))
print("\n=== Occasion contribution ===");         display(occasion_terms_factor.head(10))
print("\n=== Style    contribution ===");         display(style_terms_factor.head(10))


# Term bar charts（僅在有足夠資料時）
for term_df, col, title, fname in [
    (weather_terms_orig_full,  "weather_terms",
     "Top Weather Terms Improved by Proposed Description",
     "fig_weather_terms_orig_vs_full.pdf"),
    (occasion_terms_orig_full, "occasion_terms",
     "Top Occasion Terms Improved by Proposed Description",
     "fig_occasion_terms_orig_vs_full.pdf"),
    (style_terms_orig_full,    "style_terms",
     "Top Style Terms Improved by Proposed Description",
     "fig_style_terms_orig_vs_full.pdf"),
    (weather_terms_factor,     "weather_terms",
     "Weather Term Contribution (no_weather vs full)",
     "fig_weather_terms_factor.pdf"),
    (occasion_terms_factor,    "occasion_terms",
     "Occasion Term Contribution (no_occasion vs full)",
     "fig_occasion_terms_factor.pdf"),
    (style_terms_factor,       "style_terms",
     "Style Term Contribution (no_style vs full)",
     "fig_style_terms_factor.pdf"),
]:
    if len(term_df) > 0:
        plot_paper_category_bar(
            term_df, title=title,
            label_col=col, value_col="delta_hit10",
            top_n=8, save_name=fname,
            figsize=(7, 4)
        )


# =========================================================
# 11) Failure cases
# =========================================================
FAIL_GROUP_COLS = [
    "set_id", "target_item_id", "target_item_fg",
    "original_text", "title_full_text",
    "title_no_weather_text", "title_no_occasion_text", "title_no_style_text",
    "weather_text", "occasion_text", "style_text",
]


def aggregate_failure_candidates(df, left_name, right_name):
    out = df.copy()
    out["rank_loss"]   = out[f"rank_{right_name}"] - out[f"rank_{left_name}"]
    out["became_worse"] = (out[f"rank_{right_name}"] > out[f"rank_{left_name}"]).astype(int)

    existing = [c for c in FAIL_GROUP_COLS if c in out.columns]
    grp = out.groupby(existing, dropna=False).agg(
        n_seeds        =("seed",          "size"),
        mean_rank_left =(f"rank_{left_name}",  "mean"),
        mean_rank_right=(f"rank_{right_name}", "mean"),
        mean_rank_loss =("rank_loss",      "mean"),
        worse_count    =("became_worse",   "sum"),
    ).reset_index()

    grp["comparison"] = f"{left_name} -> {right_name}"
    grp["worse_rate"] = grp["worse_count"] / grp["n_seeds"]
    grp["category_name"] = grp["target_item_fg"].apply(map_category)

    return grp.sort_values(
        ["worse_rate", "mean_rank_loss"], ascending=[False, False]
    ).reset_index(drop=True)


fail_orig_full = aggregate_failure_candidates(m_orig_full, "orig",       "full")
fail_weather   = aggregate_failure_candidates(m_weather,   "no_weather",  "full")
fail_occasion  = aggregate_failure_candidates(m_occasion,  "no_occasion", "full")
fail_style     = aggregate_failure_candidates(m_style,     "no_style",    "full")

print("\n=== Failure cases: Original → Proposed ===")
display(fail_orig_full[["set_id","target_item_id","category_name",
                         "mean_rank_left","mean_rank_right","worse_rate"]].head(10))
print("\n=== Failure cases: Weather contribution ===")
display(fail_weather  [["set_id","target_item_id","category_name",
                         "mean_rank_left","mean_rank_right","worse_rate"]].head(10))
print("\n=== Failure cases: Occasion contribution ===")
display(fail_occasion [["set_id","target_item_id","category_name",
                         "mean_rank_left","mean_rank_right","worse_rate"]].head(10))
print("\n=== Failure cases: Style contribution ===")
display(fail_style    [["set_id","target_item_id","category_name",
                         "mean_rank_left","mean_rank_right","worse_rate"]].head(10))

# ── 建立 failure combined：full > orig（full 比 orig 差）────
# 與 success combined 相同結構，但篩選條件相反
fail_combined = combined.groupby(existing_comb, dropna=False).agg(
    n_seeds           =("seed",            "size"),
    mean_rank_orig    =("rank_orig",       "mean"),
    mean_rank_full    =("rank_full",       "mean"),
    mean_rank_weather =("rank_no_weather", "mean"),
    mean_rank_occasion=("rank_no_occasion","mean"),
    mean_rank_style   =("rank_no_style",   "mean"),
    hit10_orig        =("hit@10_orig",     "mean"),
    hit10_full        =("hit@10_full",     "mean"),
).reset_index()

# 嚴格條件：full 比 orig 差（rank 越大越差）
fail_combined = fail_combined[
    fail_combined["mean_rank_full"] > fail_combined["mean_rank_orig"]
].copy()

# 嚴重程度 = full rank - orig rank（越大越嚴重）
fail_combined["rank_loss"]     = fail_combined["mean_rank_full"] - fail_combined["mean_rank_orig"]
fail_combined["hit10_dropped"] = (
    (fail_combined["hit10_orig"] >= 0.5) & (fail_combined["hit10_full"] < 0.5)
).astype(int)
fail_combined["category_name"] = fail_combined["target_item_fg"].apply(map_category)

# 最嚴重失敗：hit 變 miss 優先，再依 rank_loss 由大到小
fail_combined = fail_combined.sort_values(
    ["hit10_dropped", "rank_loss"], ascending=[False, False]
).reset_index(drop=True)

print(f"\n⚠️  Failure combined: {len(fail_combined)} cases (full > orig)")
display(fail_combined[[
    "set_id", "target_item_id", "category_name",
    "mean_rank_orig", "mean_rank_full", "rank_loss", "hit10_dropped"
]].head(10))


# =========================================================
# 12) 儲存最終摘要表
# =========================================================
if SAVE_TABLES:
    pd.concat([
        agg_orig_full.assign(case_type="orig_to_full"),
        agg_weather  .assign(case_type="weather"),
        agg_occasion .assign(case_type="occasion"),
        agg_style    .assign(case_type="style"),
    ]).to_csv(os.path.join(TABLE_DIR, "qual_user_cases.csv"), index=False)

    pd.concat([
        fail_orig_full.assign(case_type="orig_to_full"),
        fail_weather  .assign(case_type="weather"),
        fail_occasion .assign(case_type="occasion"),
        fail_style    .assign(case_type="style"),
    ]).to_csv(os.path.join(TABLE_DIR, "qual_failure_cases.csv"), index=False)

    pd.concat([
        cat_orig_full.assign(case_type="orig_to_full"),
        cat_weather  .assign(case_type="weather"),
        cat_occasion .assign(case_type="occasion"),
        cat_style    .assign(case_type="style"),
    ]).to_csv(os.path.join(TABLE_DIR, "qual_category_summary.csv"), index=False)

    pd.concat([
        weather_terms_orig_full .assign(case_type="weather_orig_to_full"),
        occasion_terms_orig_full.assign(case_type="occasion_orig_to_full"),
        style_terms_orig_full   .assign(case_type="style_orig_to_full"),
        weather_terms_factor    .assign(case_type="weather_factor"),
        occasion_terms_factor   .assign(case_type="occasion_factor"),
        style_terms_factor      .assign(case_type="style_factor"),
    ]).to_csv(os.path.join(TABLE_DIR, "qual_condition_summary.csv"), index=False)

    print(f"[saved] tables → {TABLE_DIR}")


# =========================================================
# 13) 自動顯示各因子貢獻最大的 Top-N 個案
# =========================================================
TOP_N_CASES = 3   # 每個因子顯示幾個 case，可自行調整

FACTOR_GROUPS = [
    (agg_weather_driven,  "Weather  Factor  (no_weather  rank ≫ full rank)"),
    (agg_occasion_driven, "Occasion Factor  (no_occasion rank ≫ full rank)"),
    (agg_style_driven,    "Style    Factor  (no_style    rank ≫ full rank)"),
]

for agg_df, group_title in FACTOR_GROUPS:
    n_show = min(TOP_N_CASES, len(agg_df))
    print(f"\n{'#'*68}")
    print(f"  ▶  {group_title}")
    print(f"     有效案例數：{len(agg_df)}，顯示前 {n_show} 筆")
    print(f"{'#'*68}")
    for idx in range(n_show):
        show_full_ablation_case(agg_df, case_idx=idx, topk=TOPK)


# =========================================================
# 14) Failure case 圖片展示（full 比 orig 最差的案例）
#     版面：Target / Original（表現較好）/ Full（表現較差）
# =========================================================
def show_failure_case(fail_df, case_idx=0, topk=5, save_fig=False):
    """
    失敗案例展示（5 行，與成功案例版面相同）：
      Row 0  Query Target header
      Row 1  Target image（置中）
      Row 2  Original (Baseline)
      Row 3  Original top-k
      Row 4  No-Weather (ablation)
      Row 5  No-Weather top-k
      Row 6  No-Occasion (ablation)
      Row 7  No-Occasion top-k
      Row 8  No-Style (ablation)
      Row 9  No-Style top-k
      Row 10 Full (Proposed)  ← 紅色，表現比 orig 差
      Row 11 Full top-k
    """
    if case_idx >= len(fail_df):
        print(f"case_idx={case_idx} out of range (total={len(fail_df)})")
        return

    case     = fail_df.iloc[case_idx]
    sid      = str(case["set_id"])
    tid      = str(case["target_item_id"])
    cat_name = case["category_name"]

    # 代表 seed：rank_full 最接近 mean_rank_full（與圖片對應）
    cand = combined[
        (combined["set_id"].astype(str)         == sid) &
        (combined["target_item_id"].astype(str) == tid)
    ].copy()

    if cand.empty:
        print("No raw rows found.")
        return

    cand["_dist"] = (cand["rank_full"] - case["mean_rank_full"]).abs()
    row = cand.sort_values("_dist").iloc[0]

    def _ids(col):
        v = row.get(col)
        return (v if isinstance(v, list) else [])[:topk]

    ids_orig     = _ids("top10_ids_list_orig")
    ids_weather  = _ids("top10_ids_list_no_weather")
    ids_occasion = _ids("top10_ids_list_no_occasion")
    ids_style    = _ids("top10_ids_list_no_style")
    ids_full     = _ids("top10_ids_list_full")

    r_orig     = int(row["rank_orig"])
    r_weather  = int(row["rank_no_weather"])  if pd.notna(row.get("rank_no_weather"))  else None
    r_occasion = int(row["rank_no_occasion"]) if pd.notna(row.get("rank_no_occasion")) else None
    r_style    = int(row["rank_no_style"])    if pd.notna(row.get("rank_no_style"))    else None
    r_full     = int(row["rank_full"])

    mr_orig    = case["mean_rank_orig"]
    mr_weather = case["mean_rank_weather"]
    mr_occasion= case["mean_rank_occasion"]
    mr_style   = case["mean_rank_style"]
    mr_full    = case["mean_rank_full"]

    # ── 文字摘要 ─────────────────────────────────────────
    SEP  = "=" * 72
    DASH = "─" * 72
    print(f"\n{SEP}")
    print(f"  ⚠️  Failure Case #{case_idx}  |  set={sid}  |  target={tid}  |  [{cat_name}]")
    print(DASH)
    print(f"  {'variant':>12s}  {'seed rank':>10s}  {'mean rank':>10s}")
    print(DASH)
    for lbl, rk, mrk in [
        ("orig",       r_orig,    mr_orig),
        ("no_weather", r_weather, mr_weather),
        ("no_occasion",r_occasion,mr_occasion),
        ("no_style",   r_style,   mr_style),
        ("full",       r_full,    mr_full),
    ]:
        if rk is None:
            print(f"  {lbl:>12s}  {'—':>10s}  {mrk:>10.1f}")
            continue
        arrow = "—" if lbl == "orig" else ("▲" if rk < r_orig else "▼")
        print(f"  {lbl:>12s}  {rk:>10d}  {mrk:>10.1f}  {arrow}")
    print(DASH)
    print(f"  [Original] {str(case.get('original_text',   ''))[:110]}")
    print(f"  [Full    ] {str(case.get('title_full_text', ''))[:110]}")
    print(f"  ↳ Weather : {case.get('weather_text','')}")
    print(f"  ↳ Occasion: {case.get('occasion_text','')}")
    print(f"  ↳ Style   : {case.get('style_text','')}")
    print(f"{SEP}\n")

    # ── result rows（full 用紅色標示為失敗）─────────────
    result_rows = [
        (ids_orig,     r_orig,    mr_orig,    "Original (Baseline)",    "#555555"),
        (ids_weather,  r_weather, mr_weather, "No-Weather (ablation)",  "#C0392B"),
        (ids_occasion, r_occasion,mr_occasion,"No-Occasion (ablation)", "#E67E22"),
        (ids_style,    r_style,   mr_style,   "No-Style (ablation)",    "#8E44AD"),
        (ids_full,     r_full,    mr_full,    "Full (Proposed) ⚠️ worse","#B03A2E"),
    ]
    n_result = len(result_rows)

    # ── GridSpec ─────────────────────────────────────────
    from matplotlib.gridspec import GridSpec
    HDR_H, IMG_H, TGT_H = 0.30, 2.5, 2.2
    height_ratios = [HDR_H, TGT_H] + [HDR_H, IMG_H] * n_result
    n_gs = 2 + n_result * 2

    fig_w = max(topk, 3) * 2.45 + 0.3
    fig_h = sum(height_ratios) + 0.5

    fig = plt.figure(figsize=(fig_w, fig_h))
    gs  = GridSpec(n_gs, topk, figure=fig,
                   height_ratios=height_ratios,
                   hspace=0.0, wspace=0.07,
                   top=0.97, bottom=0.03, left=0.02, right=0.98)

    def _draw(ax, img_id, is_hit=False, is_target=False):
        img = open_image_or_none(img_id)
        if img is not None:
            ax.imshow(img)
        else:
            ax.set_facecolor("#eeeeee")
            ax.text(0.5, 0.5, "N/A", ha="center", va="center",
                    fontsize=7, color="#aaaaaa", transform=ax.transAxes)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel(
            "✓ Hit" if is_hit else "",
            fontsize=8, color="#27AE60" if is_hit else "#666666",
            fontweight="bold" if is_hit else "normal", labelpad=2
        )
        if is_target: lw, ec = 4.0, "#111111"
        elif is_hit:  lw, ec = 5.5, "#27AE60"
        else:         lw, ec = 0.7, "#cccccc"
        for sp in ax.spines.values():
            sp.set_visible(True); sp.set_edgecolor(ec); sp.set_linewidth(lw)

    def _header(gs_row, text, color):
        ax_h = fig.add_subplot(gs[gs_row, :])
        ax_h.axis("off")
        ax_h.text(0.5, 0.45, text, ha="center", va="center",
                  fontsize=10, fontweight="bold", color=color,
                  transform=ax_h.transAxes)

    # Row 0: Target header
    _header(0, f"Query Target  —  {cat_name}", "#333333")
    mid = topk // 2
    for c in range(topk):
        ax = fig.add_subplot(gs[1, c])
        if c == mid:
            _draw(ax, tid, is_target=True)
        else:
            ax.set_visible(False)

    for r, (ids, rk, mrk, label, color) in enumerate(result_rows):
        rank_str = str(rk) if rk is not None else "—"
        _header(2 + r * 2, f"{label}   (rank = {rank_str})", color)
        for i in range(topk):
            ax = fig.add_subplot(gs[2 + r * 2 + 1, i])
            if i < len(ids):
                iid = ids[i]
                _draw(ax, iid, is_hit=(str(iid) == str(tid)))
                ax.text(0.03, 0.97, f"#{i+1}", transform=ax.transAxes,
                        fontsize=7.5, va="top", ha="left", color="#444444",
                        bbox=dict(fc="white", ec="none", alpha=0.7, pad=1))
            else:
                ax.set_visible(False)

    fig.legend(
        handles=[mpatches.Patch(facecolor="#27AE60", edgecolor="#27AE60",
                                label="Ground-truth target retrieved")],
        loc="lower center", ncol=1, fontsize=8,
        frameon=False, bbox_to_anchor=(0.5, 0.0)
    )
    fig.suptitle(
        f"⚠️  Failure Case #{case_idx}  |  set={sid}  |  {cat_name}",
        fontsize=10, fontweight="bold", y=0.99
    )

    if SAVE_FIGS or save_fig:
        fname = f"failure_case_idx{case_idx:02d}.pdf"
        fpath = os.path.join(FIG_DIR, fname)
        plt.savefig(fpath, dpi=300, bbox_inches="tight")
        print(f"[saved] {fpath}")
    plt.show()


# =========================================================
# 14) 自動顯示最嚴重失敗 Top-N
# =========================================================
# =========================================================
# 14) 自動顯示最嚴重失敗 Top-N
#     每個失敗案例後接著顯示同 set 所有 item 的推薦總覽
# =========================================================
def show_set_overview(sid, topk_show=3, highlight_tid=None, save_fig=False, case_label=""):
    """
    同一 set 所有 target item 的推薦總覽。

    每個 item 一列：
      col 0          : Target item（黑框 / 黃框=當前失敗 item）
      col 1          : Orig rank 數字（文字格）
      col 2..topk+1  : Orig top-k retrieved
      col topk+2     : Full rank 數字（文字格）
      col topk+3..   : Full top-k retrieved

    命中 → 綠框 + ✓；未命中 → 淡灰框
    """
    sid = str(sid)

    set_rows = combined[combined["set_id"].astype(str) == sid].copy()
    if set_rows.empty:
        print(f"set_id={sid} 無資料")
        return

    # 每個 target_item 取代表 seed（full rank 最接近平均）
    items = []
    for tid_i, grp in set_rows.groupby("target_item_id"):
        mean_full = grp["rank_full"].mean()
        grp2 = grp.copy()
        grp2["_dist"] = (grp2["rank_full"] - mean_full).abs()
        rep = grp2.sort_values("_dist").iloc[0]

        ids_orig = (rep.get("top10_ids_list_orig") or [])[:topk_show]
        ids_full = (rep.get("top10_ids_list_full") or [])[:topk_show]
        r_orig   = int(rep["rank_orig"])
        r_full   = int(rep["rank_full"])

        items.append({
            "tid"     : str(tid_i),
            "cat"     : map_category(str(rep["target_item_fg"])),
            "ids_orig": ids_orig,
            "ids_full": ids_full,
            "r_orig"  : r_orig,
            "r_full"  : r_full,
            "hit_orig": any(str(x) == str(tid_i) for x in ids_orig),
            "hit_full": any(str(x) == str(tid_i) for x in ids_full),
        })

    # 排序：full miss 優先，再依 full rank 由大到小（最難的在上）
    items.sort(key=lambda x: (x["hit_full"], -x["r_full"]))

    n_items = len(items)
    # 欄結構：target | rank_orig | orig×topk | rank_full | full×topk
    # 用 rank 數字格代替純文字，避免多餘欄位：直接寫在 target 格的 xlabel
    # → 簡化為：target | orig×topk | full×topk，rank 寫在各區塊的 title
    n_cols = 1 + topk_show + topk_show

    cell_w, cell_h = 1.9, 1.9
    fig_w = n_cols * cell_w + 0.3
    fig_h = n_items * cell_h + 0.9

    fig, axes = plt.subplots(
        n_items, n_cols,
        figsize=(fig_w, fig_h),
        gridspec_kw={"wspace": 0.05, "hspace": 0.6}
    )
    if n_items == 1:
        axes = axes[np.newaxis, :]

    for ax in axes.flatten():
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(True); sp.set_edgecolor("#dddddd"); sp.set_linewidth(0.6)

    def _img(ax, img_id):
        img = open_image_or_none(img_id)
        if img:
            ax.imshow(img)
        else:
            ax.set_facecolor("#eeeeee")
            ax.text(0.5, 0.5, "N/A", ha="center", va="center",
                    fontsize=6, transform=ax.transAxes, color="#aaaaaa")

    def _border(ax, is_hit, is_focus=False):
        if is_focus:   lw, ec = 3.5, "#D4AC0D"
        elif is_hit:   lw, ec = 4.0, "#27AE60"
        else:          lw, ec = 0.6, "#cccccc"
        for sp in ax.spines.values():
            sp.set_edgecolor(ec); sp.set_linewidth(lw)

    for row_i, item in enumerate(items):
        tid_i    = item["tid"]
        is_focus = (str(tid_i) == str(highlight_tid))

        # ── col 0: Target ────────────────────────────────
        ax0 = axes[row_i, 0]
        _img(ax0, tid_i)
        _border(ax0, False, is_focus)
        ax0.set_title(f"{item['cat']}", fontsize=6.5, pad=2,
                      color="#D4AC0D" if is_focus else "#333333",
                      fontweight="bold" if is_focus else "normal")

        # ── col 1..topk: Orig top-k ───────────────────────
        for ci, iid in enumerate(item["ids_orig"]):
            ax = axes[row_i, 1 + ci]
            _img(ax, iid)
            is_hit = (str(iid) == str(tid_i))
            _border(ax, is_hit)
            if is_hit:
                ax.set_xlabel("✓", fontsize=9, color="#27AE60",
                              fontweight="bold", labelpad=1)
        # 未填滿格隱藏
        for ci in range(len(item["ids_orig"]), topk_show):
            axes[row_i, 1 + ci].set_visible(False)

        # Orig rank 顯示在第一個 orig 格標題
        if item["ids_orig"]:
            hit_mark = "✓" if item["hit_orig"] else "✗"
            axes[row_i, 1].set_title(
                f"Orig rank={item['r_orig']} {hit_mark}",
                fontsize=6.5, pad=2,
                color="#27AE60" if item["hit_orig"] else "#C0392B"
            )

        # ── col topk+1..2*topk: Full top-k ───────────────
        for ci, iid in enumerate(item["ids_full"]):
            ax = axes[row_i, 1 + topk_show + ci]
            _img(ax, iid)
            is_hit = (str(iid) == str(tid_i))
            _border(ax, is_hit)
            if is_hit:
                ax.set_xlabel("✓", fontsize=9, color="#27AE60",
                              fontweight="bold", labelpad=1)
        for ci in range(len(item["ids_full"]), topk_show):
            axes[row_i, 1 + topk_show + ci].set_visible(False)

        # Full rank 顯示在第一個 full 格標題
        if item["ids_full"]:
            hit_mark = "✓" if item["hit_full"] else "✗"
            axes[row_i, 1 + topk_show].set_title(
                f"Full rank={item['r_full']} {hit_mark}",
                fontsize=6.5, pad=2,
                color="#27AE60" if item["hit_full"] else "#C0392B"
            )

    # ── 欄區間標頭（畫在圖上方）────────────────────────────
    # 只在 row 0 加大標頭文字
    axes[0, 0].set_title(
        f"Target\n{items[0]['cat']}", fontsize=6.5, pad=2,
        color="#D4AC0D" if str(items[0]["tid"]) == str(highlight_tid) else "#333333"
    )
    # 在第一列的中間格加區間標頭
    mid_orig = 1 + topk_show // 2
    mid_full = 1 + topk_show + topk_show // 2

    orig_hit_total = sum(1 for x in items if x["hit_orig"])
    full_hit_total = sum(1 for x in items if x["hit_full"])
    total = len(items)

    fig.text(
        (mid_orig) / n_cols, 1.0,
        f"Original  Hit@{topk_show}={orig_hit_total}/{total}",
        ha="center", va="bottom", fontsize=9,
        fontweight="bold", color="#555555",
        transform=fig.transFigure
    )
    fig.text(
        (mid_full) / n_cols, 1.0,
        f"Full (Proposed)  Hit@{topk_show}={full_hit_total}/{total}",
        ha="center", va="bottom", fontsize=9,
        fontweight="bold", color="#2166AC",
        transform=fig.transFigure
    )

    # 分隔線
    sep_x = (1 + topk_show) / n_cols
    fig.add_artist(plt.Line2D(
        [sep_x, sep_x], [0, 1],
        transform=fig.transFigure,
        color="#bbbbbb", linewidth=1, linestyle="--"
    ))

    legend_handles = [
        mpatches.Patch(facecolor="white", edgecolor="#27AE60", linewidth=3,
                       label=f"Hit (in top-{topk_show})"),
        mpatches.Patch(facecolor="white", edgecolor="#cccccc", linewidth=1,
                       label="Miss"),
        mpatches.Patch(facecolor="white", edgecolor="#D4AC0D", linewidth=3,
                       label="Current failure item"),
    ]
    fig.legend(handles=legend_handles, loc="lower center", ncol=3,
               fontsize=7.5, frameon=False, bbox_to_anchor=(0.5, -0.02))

    fig.suptitle(
        f"Set {sid}  —  All Items Retrieval Overview"
        + (f"  [{case_label}]" if case_label else ""),
        fontsize=9.5, fontweight="bold", y=1.03
    )
    plt.tight_layout()

    if SAVE_FIGS or save_fig:
        fname = f"set_overview_{sid}.pdf"
        fpath = os.path.join(FIG_DIR, fname)
        plt.savefig(fpath, dpi=300, bbox_inches="tight")
        print(f"[saved] {fpath}")
    plt.show()


# =========================================================
# 14) 自動顯示最嚴重失敗 Top-N
#     每個失敗案例後接著顯示同 set 全品項推薦總覽
# =========================================================
TOP_N_FAIL = 3   # 顯示幾個失敗案例，可自行調整

print(f"\n{'#'*68}")
print(f"  ⚠️   Failure Case Study  —  Top-{TOP_N_FAIL} Worst Cases")
print(f"      篩選條件：full rank > orig rank（共 {len(fail_combined)} 筆）")
print(f"      排序：hit→miss 優先，再依 rank_loss 由大到小")
print(f"{'#'*68}")

for idx in range(min(TOP_N_FAIL, len(fail_combined))):
    case = fail_combined.iloc[idx]
    # 1. 失敗案例 5 行詳細圖（含三個消融）
    show_failure_case(fail_combined, case_idx=idx, topk=TOPK)
    # 2. 同 set 全品項推薦總覽（orig vs full 對比）
    show_set_overview(
        sid=case["set_id"],
        topk_show=3,
        highlight_tid=case["target_item_id"],
        case_label=f"failure case #{idx}"
    )

print("\n✅ 全部個案展示完成")

[categories.csv] 載入 159 個類別，範例：{'2': 'undefined', '3': 'dress (all-body)', '4': 'dress (all-body)', '5': 'dress (all-body)'}
Loaded rows:
  original    : 17160
  full        : 17160
  no_weather  : 17160
  no_occasion : 17160
  no_style    : 17160
Loaded original titles : 68306


,set_id,original_text,original_title_only,original_url_part
0,219779031,spring trend colored denim Denim Jumpers,Denim Jumpers,spring trend colored denim
1,211158670,boot scootin outfit only 2500 Stacked-Heel Boots,Stacked-Heel Boots,boot scootin outfit only 2500
2,222049137,"mermaid for life ""Mermaid Hair, Don't Care""","""Mermaid Hair, Don't Care""",mermaid for life


Loaded fragments       : 35140


,set_id,title_full_text,weather_terms,occasion_terms,style_terms,weather_text,occasion_text,style_text,title_no_weather_text,title_no_occasion_text,title_no_style_text
0,201033839,"19°C, Effortless Mint & Lace Ensemble for a Br...","[19°C, Breezy Spring Day]",[],[Effortless Mint & Lace Ensemble],19°C | Breezy Spring Day,,Effortless Mint & Lace Ensemble,Effortless Mint & Lace Ensemble,"19°C, Effortless Mint & Lace Ensemble Breezy S...","19°C, Breezy Spring Day"
1,205002070,"24°C, Effortless Boho Vibes for Warm Days","[24°C, Warm Days]",[],[Effortless Boho Vibes],24°C | Warm Days,,Effortless Boho Vibes,Effortless Boho Vibes,"24°C, Effortless Boho Vibes Warm Days","24°C, Warm Days"
2,217778087,"17°C, Effortless Evening Glam with a Cozy Flar...",[17°C],[Evening],"[Effortless, Glam, Cozy, Flare Coat]",17°C,Evening,Effortless | Glam | Cozy | Flare Coat,Effortless Evening Glam Cozy Flare Coat,"17°C, Effortless Glam Cozy Flare Coat","17°C, Evening"


Merged rows:
  orig→full       : 17160
  weather→full    : 17160
  occasion→full   : 17160
  style→full      : 17160

  Weather-driven  cases: 62
  Occasion-driven cases: 99
  Style-driven    cases: 417


...


  ⚠️  Failure Case #2  |  set=195850285  |  target=136445502  |  [clutch (bags)]
────────────────────────────────────────────────────────────────────────
       variant   seed rank   mean rank
────────────────────────────────────────────────────────────────────────
          orig          19        12.0  —
    no_weather          61        86.4  ▼
   no_occasion          44        41.2  ▼
      no_style          25        30.2  ▼
          full          71        94.4  ▼
────────────────────────────────────────────────────────────────────────
  [Original] spring date pretty plus-size style
  [Full    ] 24.6° C, Effortless spring date look with sparkle and a touch of floral charm
  ↳ Weather : 24.6° C | spring
  ↳ Occasion: date
  ↳ Style   : Effortless spring date look | sparkle | floral charm



<Figure size 2008x2720 with 36 Axes>

/tmp/ipykernel_5560/1419879965.py:1422: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


<Figure size 2176x448 with 7 Axes>


✅ 全部個案展示完成


In [36]:
import glob, os

# 掃描 EXP_ROOT 下所有 csv，看看有哪些檔名
all_csvs = glob.glob(os.path.join(EXP_ROOT, "**/*.csv"), recursive=True)
for f in sorted(all_csvs):
    print(f)

./experiments_none_only/detail_cir/detail_cir_NewoutfitUrlTitle_subset_none_subset_seed1.csv
./experiments_none_only/detail_cir/detail_cir_NewoutfitUrlTitle_subset_none_subset_seed2.csv
./experiments_none_only/detail_cir/detail_cir_NewoutfitUrlTitle_subset_none_subset_seed3.csv
./experiments_none_only/detail_cir/detail_cir_NewoutfitUrlTitle_subset_none_subset_seed4.csv
./experiments_none_only/detail_cir/detail_cir_NewoutfitUrlTitle_subset_none_subset_seed5.csv
./experiments_none_only/detail_cir/detail_cir_no_occasion_subset_none_subset_seed1.csv
./experiments_none_only/detail_cir/detail_cir_no_occasion_subset_none_subset_seed2.csv
./experiments_none_only/detail_cir/detail_cir_no_occasion_subset_none_subset_seed3.csv
./experiments_none_only/detail_cir/detail_cir_no_occasion_subset_none_subset_seed4.csv
./experiments_none_only/detail_cir/detail_cir_no_occasion_subset_none_subset_seed5.csv
./experiments_none_only/detail_cir/detail_cir_no_style_subset_none_subset_seed1.csv
./experiments_no

In [5]:
import os
import re
import json
import ast
import glob
import textwrap
from pathlib import Path
from functools import lru_cache
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

try:
    from IPython.display import display
except Exception:
    def display(x):
        if isinstance(x, pd.DataFrame):
            print(x.to_string(index=False))
        else:
            print(x)


# =========================================================
# 0) CONFIG
# =========================================================
EXP_ROOT = "./experiments_none_only"
DETAIL_DIR = os.path.join(EXP_ROOT, "detail_cir")
if not os.path.exists(DETAIL_DIR):
    DETAIL_DIR = EXP_ROOT

DETAIL_RUN_TAG = "none_subset"
DETAIL_SUBSET_TAG = "subset"

FRAG_JSONL = "/home/ester/valid_description/wos_split_results_v5_merged_retry_round3.jsonl"
ORIGINAL_TITLE_JSON = "/home/ester/valid_description/polyvore_data/polyvore_outfits/polyvore_outfit_titles.json"
CATEGORY_CSV = "/home/ester/valid_description/polyvore_data/polyvore_outfits/categories.csv"

IMAGE_ROOTS = [
    "/home/ester/valid_description/polyvore_data/polyvore_outfits/images",
]

TOPK = 5
DISPLAY_TOPN = 3

COMPARE_SAME_SINGLE_COLOR_ONLY = True
REQUIRE_TARGET_MATCH_QUERY = True
MIN_ALT_ADVANTAGE = 2

# ---- Clear case thresholds (for visualization only) ----
CLEAR_ORIG_ISSUE_RATE_MIN = 0.60
CLEAR_FULL_ISSUE_RATE_MAX = 0.20
CLEAR_FULL_ISSUE_RATE_MIN = 0.60
CLEAR_ORIG_ALT_ADV_MIN = 1.50
CLEAR_FULL_ALT_ADV_MAX = 0.20
CLEAR_FULL_ALT_ADV_MIN = 1.50
CLEAR_QUERY_COUNT_GAIN_MIN = 1.0

SAVE_SUMMARY_CSV = True
SAVE_FIGS = False

OUTPUT_DIR = "./color_bias_compare_outputs"
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

VARIANTS = {
    "original": "outfitUrlTitle",
    "full": "NewoutfitUrlTitle",
}

COLOR_PATTERNS = {
    "black": [
        r"\bblack\b",
        r"\bjet black\b",
        r"\bcharcoal\b",
    ],
    "white": [
        r"\bwhite\b",
        r"\bivory\b",
        r"\bcream\b",
        r"\boff[- ]white\b",
    ],
    "gray": [
        r"\bgray\b",
        r"\bgrey\b",
        r"\bsilver\b",
        r"\bslate\b",
    ],
    "brown": [
        r"\bbrown\b",
        r"\bbeige\b",
        r"\btan\b",
        r"\bkhaki\b",
        r"\bcamel\b",
        r"\bnude\b",
        r"\bmocha\b",
        r"\bcognac\b",
    ],
    "red": [
        r"\bred\b",
        r"\bcrimson\b",
        r"\bburgundy\b",
        r"\bmaroon\b",
        r"\bwine\b",
    ],
    "orange": [
        r"\borange\b",
        r"\bcoral\b",
        r"\bpeach\b",
        r"\brust\b",
    ],
    "yellow": [
        r"\byellow\b",
        r"\bgold\b",
        r"\bmustard\b",
    ],
    "green": [
        r"\bgreen\b",
        r"\bolive\b",
        r"\bemerald\b",
        r"\bmint\b",
        r"\blime\b",
    ],
    "blue": [
        r"\bblue\b",
        r"\bnavy\b",
        r"\bnavy blue\b",
        r"\bsky blue\b",
        r"\blight blue\b",
        r"\bbaby blue\b",
        r"\bcobalt\b",
        r"\broyal blue\b",
        r"\bteal\b",
        r"\bturquoise\b",
        r"\bcyan\b",
    ],
    "purple": [
        r"\bpurple\b",
        r"\blavender\b",
        r"\bviolet\b",
        r"\blilac\b",
    ],
    "pink": [
        r"\bpink\b",
        r"\brose\b",
        r"\bfuchsia\b",
        r"\bmagenta\b",
        r"\bblush\b",
        r"\bhot pink\b",
    ],
}

CANONICAL_COLOR_ORDER = [
    "black", "white", "gray", "brown",
    "red", "orange", "yellow", "green",
    "blue", "purple", "pink"
]

plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 10,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
})


# =========================================================
# 1) Basic utilities
# =========================================================
def normalize_text(x):
    if x is None:
        return ""
    if isinstance(x, str):
        return x.strip()
    if isinstance(x, list):
        return " | ".join(str(i).strip() for i in x if str(i).strip())
    return str(x).strip()


def parse_top_ids(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return [str(i) for i in x]
    try:
        v = json.loads(x)
        if isinstance(v, list):
            return [str(i) for i in v]
    except Exception:
        pass
    try:
        v = ast.literal_eval(x)
        if isinstance(v, list):
            return [str(i) for i in v]
    except Exception:
        pass
    return []


def ensure_list(x):
    if isinstance(x, list):
        return x
    return []


def load_category_map(csv_path):
    if not os.path.exists(csv_path):
        return {}

    df_raw = pd.read_csv(csv_path, header=None)
    first_val = str(df_raw.iloc[0, 0]).strip()

    if first_val.isdigit():
        df = df_raw.copy()
    else:
        df = pd.read_csv(csv_path)
        id_candidates = ["id", "category_id", "fg"]
        name_candidates = ["name", "category_name", "category"]
        grp_candidates = ["group", "supercategory", "type", "parent"]

        id_col = next((c for c in id_candidates if c in df.columns), df.columns[0])
        nm_col = next((c for c in name_candidates if c in df.columns), df.columns[1])
        grp_col = next((c for c in grp_candidates if c in df.columns), None)

        df = df.rename(columns={id_col: 0, nm_col: 1})
        if grp_col:
            df = df.rename(columns={grp_col: 2})

    result = {}
    has_group = df.shape[1] >= 3
    for _, row in df.iterrows():
        cid = str(row[0]).strip()
        if cid in result:
            continue
        name = str(row[1]).strip() if pd.notna(row[1]) else cid
        if has_group and pd.notna(row[2]):
            grp = str(row[2]).strip()
            result[cid] = f"{name} ({grp})"
        else:
            result[cid] = name
    return result


CATEGORY_MAP = load_category_map(CATEGORY_CSV)


def map_category(x):
    return CATEGORY_MAP.get(str(x), str(x))


def extract_query_colors(text):
    text = normalize_text(text).lower()
    found = []
    for canonical in CANONICAL_COLOR_ORDER:
        for pat in COLOR_PATTERNS[canonical]:
            if re.search(pat, text):
                found.append(canonical)
                break
    return list(dict.fromkeys(found))


def find_image_by_id(img_id):
    img_id = str(img_id)
    for root in IMAGE_ROOTS:
        for ext in (".jpg", ".jpeg", ".png", ".webp"):
            p = Path(root) / f"{img_id}{ext}"
            if p.exists():
                return str(p)
    return None


def open_image_or_none(img_id):
    fp = find_image_by_id(img_id)
    if fp is None:
        return None
    try:
        return Image.open(fp).convert("RGB")
    except Exception:
        return None


def short_profile_text(profile, topn=2):
    if not profile:
        return "unknown"
    return ", ".join(f"{c}({p:.2f})" for c, p in profile[:topn])


def safe_mode_non_none(series, default="none"):
    vals = [x for x in series if pd.notna(x) and x != "none"]
    if not vals:
        return default
    return Counter(vals).most_common(1)[0][0]


def wrap_text(s, width=110):
    s = normalize_text(s)
    if not s:
        return ""
    return textwrap.fill(s, width=width)


def format_rank_text(rank_value):
    if pd.isna(rank_value):
        return "Correct answer rank: N/A"
    return f"Correct answer rank: #{int(round(float(rank_value)))}"


# =========================================================
# 2) Image dominant-color analysis
# =========================================================
@lru_cache(maxsize=None)
def get_image_color_profile(img_id):
    fp = find_image_by_id(img_id)
    if fp is None:
        return {"dominant": "unknown", "profile": [], "path": None}

    try:
        img_rgba = Image.open(fp).convert("RGBA")
    except Exception:
        return {"dominant": "unknown", "profile": [], "path": fp}

    try:
        resample = Image.Resampling.LANCZOS
    except Exception:
        resample = Image.LANCZOS

    img_rgba.thumbnail((120, 120), resample)

    arr = np.array(img_rgba)
    rgb = arr[..., :3]
    alpha = arr[..., 3].astype(np.float32) / 255.0

    hsv = np.array(Image.fromarray(rgb).convert("HSV")).astype(np.float32)
    h = hsv[..., 0] * (360.0 / 255.0)
    s = hsv[..., 1] / 255.0
    v = hsv[..., 2] / 255.0

    mask = alpha > 0.20
    mask &= ~((v > 0.97) & (s < 0.08))

    if mask.sum() < 50:
        mask = alpha > 0.20

    h = h[mask]
    s = s[mask]
    v = v[mask]

    if len(h) == 0:
        return {"dominant": "unknown", "profile": [], "path": fp}

    labels = np.array(["unknown"] * len(h), dtype=object)

    black_mask = v < 0.18
    white_mask = (v > 0.88) & (s < 0.12)
    gray_mask = (s < 0.18) & (~black_mask) & (~white_mask)

    labels[black_mask] = "black"
    labels[white_mask] = "white"
    labels[gray_mask] = "gray"

    rest = labels == "unknown"
    brown_mask = rest & (h >= 10) & (h < 40) & (s >= 0.20) & (v < 0.65)
    labels[brown_mask] = "brown"

    rest = labels == "unknown"
    labels[rest & ((h < 15) | (h >= 345))] = "red"

    rest = labels == "unknown"
    labels[rest & (h >= 15) & (h < 40)] = "orange"

    rest = labels == "unknown"
    labels[rest & (h >= 40) & (h < 65)] = "yellow"

    rest = labels == "unknown"
    labels[rest & (h >= 65) & (h < 170)] = "green"

    rest = labels == "unknown"
    labels[rest & (h >= 170) & (h < 255)] = "blue"

    rest = labels == "unknown"
    labels[rest & (h >= 255) & (h < 290)] = "purple"

    rest = labels == "unknown"
    labels[rest & (h >= 290) & (h < 345)] = "pink"

    cnt = Counter(labels.tolist())
    cnt.pop("unknown", None)

    if len(cnt) == 0:
        return {"dominant": "unknown", "profile": [], "path": fp}

    total = sum(cnt.values())
    profile = sorted(
        [(k, v / total) for k, v in cnt.items()],
        key=lambda x: x[1],
        reverse=True
    )

    return {
        "dominant": profile[0][0],
        "profile": profile,
        "path": fp,
    }


# =========================================================
# 3) Load detail / metadata
# =========================================================
def load_one_variant_detail(variant_name, short_name):
    pattern = os.path.join(
        DETAIL_DIR,
        f"detail_cir_{variant_name}_{DETAIL_SUBSET_TAG}_{DETAIL_RUN_TAG}_seed*.csv"
    )
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"Cannot find detail files: {pattern}")

    dfs = []
    for fp in files:
        df = pd.read_csv(fp)
        df["variant"] = short_name
        dfs.append(df)

    out = pd.concat(dfs, ignore_index=True)

    required = ["seed", "set_id", "target_item_id", "target_item_fg", "rank", "top10_ids"]
    for c in required:
        if c not in out.columns:
            raise KeyError(f"Missing required column in detail csv: {c}")

    out["seed"] = out["seed"].astype(int)
    out["set_id"] = out["set_id"].astype(str)
    out["target_item_id"] = out["target_item_id"].astype(str)
    out["target_item_fg"] = out["target_item_fg"].astype(str)
    out["rank"] = pd.to_numeric(out["rank"], errors="coerce")
    out["top10_ids_list"] = out["top10_ids"].apply(parse_top_ids)
    out["query_key"] = out["seed"].astype(str) + "||" + out["set_id"] + "||" + out["target_item_id"]
    return out


def load_original_titles_json(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Cannot find original title file: {path}")

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []
    for sid, value in data.items():
        if isinstance(value, str):
            original_text = value.strip()
        elif isinstance(value, dict):
            title_part = normalize_text(value.get("title", ""))
            url_part = normalize_text(value.get("url_name", value.get("url", "")))
            original_text = " ".join(p for p in [url_part, title_part] if p).strip()
        else:
            original_text = normalize_text(value)

        rows.append({
            "set_id": str(sid),
            "original_text": original_text,
        })
    return pd.DataFrame(rows)


def load_fragments_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Cannot find fragments file: {path}")

    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if obj.get("valid") is not True:
                continue

            title_ablation = obj.get("title_ablation", {})
            rows.append({
                "set_id": str(obj.get("id", "")),
                "title_full_text": normalize_text(obj.get("title", "")),
                "title_no_weather_text": normalize_text(title_ablation.get("no_weather", "")),
                "title_no_occasion_text": normalize_text(title_ablation.get("no_occasion", "")),
                "title_no_style_text": normalize_text(title_ablation.get("no_style", "")),
            })
    return pd.DataFrame(rows)


# =========================================================
# 4) Row-level issue detection
# =========================================================
def analyze_variant_row(row, variant_name):
    query_color = row[f"query_color_{variant_name}"]
    target_id = str(row["target_item_id"])
    top_ids = ensure_list(row[f"top10_ids_list_{variant_name}"])[:TOPK]

    target_info = get_image_color_profile(target_id)
    target_top2 = [c for c, _ in target_info["profile"][:2]]
    target_match = int(query_color in target_top2)

    top_infos = [get_image_color_profile(str(i)) for i in top_ids]
    top_colors = [x["dominant"] for x in top_infos]
    cnt = Counter([c for c in top_colors if c != "unknown"])

    query_count = cnt.get(query_color, 0)
    non_query = {k: v for k, v in cnt.items() if k != query_color}

    if non_query:
        wrong_color, wrong_count = sorted(non_query.items(), key=lambda x: (-x[1], x[0]))[0]
    else:
        wrong_color, wrong_count = "none", 0

    alt_advantage = wrong_count - query_count
    issue = int(alt_advantage >= MIN_ALT_ADVANTAGE)

    if REQUIRE_TARGET_MATCH_QUERY:
        issue = int(issue and (target_match == 1))

    return pd.Series({
        f"target_match_{variant_name}": target_match,
        f"target_profile_text_{variant_name}": short_profile_text(target_info["profile"], topn=2),
        f"query_count_{variant_name}": query_count,
        f"wrong_color_{variant_name}": wrong_color,
        f"wrong_count_{variant_name}": wrong_count,
        f"alt_advantage_{variant_name}": alt_advantage,
        f"topk_colors_{variant_name}": top_colors,
        f"topk_color_count_text_{variant_name}": ", ".join([f"{k}:{v}" for k, v in cnt.most_common()]) if cnt else "none",
        f"issue_{variant_name}": issue,
    })


# =========================================================
# 5) Load full outfit items
# =========================================================
def parse_outfit_item_ids(record):
    keys = ["items", "item_ids", "itemIdList", "item_list", "item_ids_list"]
    items = None
    for k in keys:
        if k in record:
            items = record[k]
            break

    if items is None:
        return None

    out = []
    if isinstance(items, list):
        for x in items:
            if isinstance(x, dict):
                iid = None
                for kk in ["item_id", "itemId", "id", "index"]:
                    if kk in x:
                        iid = x[kk]
                        break
                if iid is not None:
                    out.append(str(iid))
            else:
                out.append(str(x))

    return out if out else None


def load_outfit_map():
    search_roots = list({
        str(Path(ORIGINAL_TITLE_JSON).parent),
        str(Path(ORIGINAL_TITLE_JSON).parent.parent)
        if Path(ORIGINAL_TITLE_JSON).parent.parent != Path(ORIGINAL_TITLE_JSON).parent
        else str(Path(ORIGINAL_TITLE_JSON).parent),
    })

    candidate_files = []
    patterns = [
        "*outfit*.json",
        "train.json",
        "valid.json",
        "val.json",
        "test.json",
        "*/*.json",
        "*/*/*.json",
    ]

    for root in search_roots:
        if not os.path.exists(root):
            continue
        for pat in patterns:
            candidate_files.extend(glob.glob(os.path.join(root, pat)))

    candidate_files = sorted(set(candidate_files))
    skip_names = {"polyvore_outfit_titles.json", "categories.json", "categories.csv"}

    outfit_map = {}

    for fp in candidate_files:
        base = os.path.basename(fp)
        if base in skip_names:
            continue

        try:
            with open(fp, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception:
            continue

        if isinstance(data, list):
            records = data
        elif isinstance(data, dict) and all(isinstance(v, dict) for v in data.values()):
            records = list(data.values())
        else:
            continue

        added = 0
        for rec in records:
            if not isinstance(rec, dict):
                continue

            sid = rec.get("set_id", rec.get("setId", rec.get("id")))
            if sid is None:
                continue

            item_ids = parse_outfit_item_ids(rec)
            if item_ids:
                outfit_map[str(sid)] = item_ids
                added += 1

        if added > 0:
            print(f"[outfit map] loaded {added} sets from {fp}")

    print(f"[outfit map] total sets loaded: {len(outfit_map)}")
    return outfit_map


# =========================================================
# 6) Visualization
# =========================================================
def draw_img(ax, img_id, title="", border_color="#CCCCCC", label_text=""):
    img = open_image_or_none(img_id)
    if img is not None:
        ax.imshow(img)
    else:
        ax.set_facecolor("#EEEEEE")
        ax.text(0.5, 0.5, "N/A", ha="center", va="center", fontsize=8, color="#999999")

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=8, pad=2)
    ax.set_xlabel(label_text, fontsize=8, labelpad=2)

    for sp in ax.spines.values():
        sp.set_visible(True)
        sp.set_edgecolor(border_color)
        sp.set_linewidth(3.0 if border_color != "#CCCCCC" else 0.8)


def show_case(case_row, case_type, outfit_map):
    sid = str(case_row["set_id"])
    tid = str(case_row["target_item_id"])
    cat = map_category(case_row["target_item_fg"])

    outfit_ids = outfit_map.get(sid, [])
    if not outfit_ids:
        outfit_ids = [tid]

    orig_topk = ensure_list(case_row["rep_topk_ids_original"])[:TOPK]
    full_topk = ensure_list(case_row["rep_topk_ids_full"])[:TOPK]
    orig_colors = ensure_list(case_row["rep_topk_colors_original"])[:TOPK]
    full_colors = ensure_list(case_row["rep_topk_colors_full"])[:TOPK]

    # representative row rank; fallback to mean rank
    orig_rank_value = case_row["rep_rank_original"] if "rep_rank_original" in case_row.index else np.nan
    full_rank_value = case_row["rep_rank_full"] if "rep_rank_full" in case_row.index else np.nan

    if pd.isna(orig_rank_value):
        orig_rank_value = case_row.get("mean_rank_original", np.nan)
    if pd.isna(full_rank_value):
        full_rank_value = case_row.get("mean_rank_full", np.nan)

    orig_rank_text = format_rank_text(orig_rank_value)
    full_rank_text = format_rank_text(full_rank_value)

    ncols = max(len(outfit_ids), TOPK, 3)

    fig, axes = plt.subplots(
        3, ncols,
        figsize=(ncols * 2.2, 7.8),
        gridspec_kw={"height_ratios": [1.25, 1.0, 1.0], "hspace": 0.40}
    )

    if ncols == 1:
        axes = np.array([[axes[0]], [axes[1]], [axes[2]]])

    # 留比較多空間給上方 query 與下方 rank
    fig.subplots_adjust(left=0.14, right=0.98, top=0.83, bottom=0.12)

    # ----------------------------
    # Row 1: full outfit
    # ----------------------------
    for c in range(ncols):
        if c < len(outfit_ids):
            iid = outfit_ids[c]
            border = "#D4AC0D" if str(iid) == tid else "#CCCCCC"
            label = "TARGET SLOT" if str(iid) == tid else ""
            draw_img(
                axes[0, c],
                iid,
                title=f"Item {c+1}",
                border_color=border,
                label_text=label
            )
        else:
            axes[0, c].axis("off")

    # ----------------------------
    # Row 2: original retrieval
    # ----------------------------
    for c in range(ncols):
        if c < len(orig_topk):
            col = orig_colors[c] if c < len(orig_colors) else "unknown"
            if col == case_row["query_color"]:
                border = "#2E8B57"
            elif col == case_row["wrong_color_mode_original"]:
                border = "#C0392B"
            else:
                border = "#CCCCCC"

            if str(orig_topk[c]) == tid:
                border = "#D4AC0D"

            draw_img(
                axes[1, c],
                orig_topk[c],
                title=f"Original #{c+1}",
                border_color=border,
                label_text=col
            )
        else:
            axes[1, c].axis("off")

    # ----------------------------
    # Row 3: full retrieval
    # ----------------------------
    for c in range(ncols):
        if c < len(full_topk):
            col = full_colors[c] if c < len(full_colors) else "unknown"
            if col == case_row["query_color"]:
                border = "#2E8B57"
            elif col == case_row["wrong_color_mode_full"]:
                border = "#C0392B"
            else:
                border = "#CCCCCC"

            if str(full_topk[c]) == tid:
                border = "#D4AC0D"

            draw_img(
                axes[2, c],
                full_topk[c],
                title=f"Full #{c+1}",
                border_color=border,
                label_text=col
            )
        else:
            axes[2, c].axis("off")

    # 先讓版面穩定下來，才能讀正確座標
    fig.canvas.draw()

    # 取得三排的實際位置
    row0_pos = axes[0, 0].get_position()
    row1_pos = axes[1, 0].get_position()
    row2_pos = axes[2, 0].get_position()

    row0_center_y = (row0_pos.y0 + row0_pos.y1) / 2
    row1_center_y = (row1_pos.y0 + row1_pos.y1) / 2
    row2_center_y = (row2_pos.y0 + row2_pos.y1) / 2

    # rank 文字放在該排圖片下方，不再用固定 y
    orig_rank_y = max(row1_pos.y0 - 0.045, row2_pos.y1 + 0.015)
    full_rank_y = max(row2_pos.y0 - 0.055, 0.03)

    # ----------------------------
    # Titles / texts
    # ----------------------------
    fig.suptitle(
        f"[{case_type}] set={sid} | target={tid} | {cat} | query_color={case_row['query_color']}",
        fontsize=12,
        fontweight="bold",
        y=0.985
    )

    # 上方描述保留，但更高
    fig.text(
        0.01, 0.935,
        f"Original query: {wrap_text(case_row['query_text_original'], 120)}",
        fontsize=9, va="top"
    )
    fig.text(
        0.01, 0.895,
        f"Full query: {wrap_text(case_row['query_text_full'], 120)}",
        fontsize=9, va="top"
    )

    # row labels：跟著每排中心走，永遠不會壓圖
    fig.text(0.01, row0_center_y, "Full outfit", fontsize=10, fontweight="bold", va="center")
    fig.text(0.01, row1_center_y, "Original retrieval", fontsize=10, fontweight="bold", va="center")
    fig.text(0.01, row2_center_y, "Full retrieval", fontsize=10, fontweight="bold", va="center")

    # rank：放在各排下方
    fig.text(
        0.50, orig_rank_y,
        orig_rank_text,
        ha="center", va="center",
        fontsize=10, fontweight="bold", color="#444444"
    )
    fig.text(
        0.50, full_rank_y,
        full_rank_text,
        ha="center", va="center",
        fontsize=10, fontweight="bold", color="#2166AC"
    )

    if SAVE_FIGS:
        fname = os.path.join(FIG_DIR, f"{case_type}_set{sid}_target{tid}.png")
        plt.savefig(fname, dpi=220, bbox_inches="tight")
        print(f"[saved] {fname}")

    plt.show()


# =========================================================
# 7) Main
# =========================================================
def main():
    print("=" * 80)
    print("1) Load detail csv")
    print("=" * 80)
    detail_orig = load_one_variant_detail(VARIANTS["original"], "original")
    detail_full = load_one_variant_detail(VARIANTS["full"], "full")
    print("detail_original:", len(detail_orig))
    print("detail_full    :", len(detail_full))

    print("\n" + "=" * 80)
    print("2) Load metadata")
    print("=" * 80)
    orig_title_df = load_original_titles_json(ORIGINAL_TITLE_JSON)
    frag_df = load_fragments_jsonl(FRAG_JSONL)

    base = (
        detail_orig[["query_key", "seed", "set_id", "target_item_id", "target_item_fg", "rank", "top10_ids_list"]]
        .rename(columns={"rank": "rank_original", "top10_ids_list": "top10_ids_list_original"})
        .merge(
            detail_full[["query_key", "rank", "top10_ids_list"]]
            .rename(columns={"rank": "rank_full", "top10_ids_list": "top10_ids_list_full"}),
            on="query_key",
            how="inner"
        )
        .merge(orig_title_df, on="set_id", how="left")
        .merge(frag_df, on="set_id", how="left")
    )

    base["query_text_original"] = base["original_text"].fillna("").astype(str)
    base["query_text_full"] = base["title_full_text"].fillna("").astype(str)

    base["query_colors_original"] = base["query_text_original"].apply(extract_query_colors)
    base["query_colors_full"] = base["query_text_full"].apply(extract_query_colors)

    if COMPARE_SAME_SINGLE_COLOR_ONLY:
        base = base[
            (base["query_colors_original"].apply(lambda x: len(x) == 1)) &
            (base["query_colors_full"].apply(lambda x: len(x) == 1)) &
            (base["query_colors_original"].str[0] == base["query_colors_full"].str[0])
        ].copy()
        base["query_color"] = base["query_colors_original"].str[0]
    else:
        base = base[
            (base["query_colors_original"].apply(lambda x: len(x) >= 1)) &
            (base["query_colors_full"].apply(lambda x: len(x) >= 1))
        ].copy()
        base["query_color"] = base["query_colors_full"].str[0]

    base["query_color_original"] = base["query_color"]
    base["query_color_full"] = base["query_color"]

    print("comparable rows:", len(base))
    if len(base) == 0:
        print("No comparable color-query rows found.")
        return

    needed_ids = set(base["target_item_id"].astype(str).tolist())
    for col in ["top10_ids_list_original", "top10_ids_list_full"]:
        for ids in base[col]:
            needed_ids.update([str(x) for x in ensure_list(ids)[:TOPK]])

    print("need analyze images:", len(needed_ids))
    for i, iid in enumerate(sorted(needed_ids), 1):
        _ = get_image_color_profile(iid)
        if i % 500 == 0:
            print(f"  cached {i}/{len(needed_ids)}")

    print("\n" + "=" * 80)
    print("3) Analyze color-bias issue in original / full")
    print("=" * 80)

    base = base.reset_index(drop=True)

    ana_orig = base.apply(
        analyze_variant_row, axis=1, variant_name="original"
    ).reset_index(drop=True)

    ana_full = base.apply(
        analyze_variant_row, axis=1, variant_name="full"
    ).reset_index(drop=True)

    row_df = pd.concat([base, ana_orig, ana_full], axis=1)

    print("NaN in alt_advantage_original:", row_df["alt_advantage_original"].isna().sum())
    print("NaN in alt_advantage_full    :", row_df["alt_advantage_full"].isna().sum())

    row_df["solved_row"] = ((row_df["issue_original"] == 1) & (row_df["issue_full"] == 0)).astype(int)
    row_df["unsolved_row"] = ((row_df["issue_original"] == 1) & (row_df["issue_full"] == 1)).astype(int)
    row_df["case_key"] = row_df["set_id"].astype(str) + "||" + row_df["target_item_id"].astype(str)

    rep_source = (
        row_df
        .dropna(subset=["alt_advantage_original"])
        .sort_values(
            ["case_key", "alt_advantage_original", "rank_original"],
            ascending=[True, False, True]
        )
        .drop_duplicates(subset=["case_key"], keep="first")
    )

    if len(rep_source) > 0:
        rep_df = rep_source[[
            "case_key",
            "rank_original",
            "rank_full",
            "top10_ids_list_original",
            "top10_ids_list_full",
            "topk_colors_original",
            "topk_colors_full",
            "topk_color_count_text_original",
            "topk_color_count_text_full",
        ]].copy().rename(columns={
            "rank_original": "rep_rank_original",
            "rank_full": "rep_rank_full",
            "top10_ids_list_original": "rep_topk_ids_original",
            "top10_ids_list_full": "rep_topk_ids_full",
            "topk_colors_original": "rep_topk_colors_original",
            "topk_colors_full": "rep_topk_colors_full",
            "topk_color_count_text_original": "rep_topk_color_count_text_original",
            "topk_color_count_text_full": "rep_topk_color_count_text_full",
        })
    else:
        rep_df = pd.DataFrame(columns=[
            "case_key",
            "rep_rank_original",
            "rep_rank_full",
            "rep_topk_ids_original",
            "rep_topk_ids_full",
            "rep_topk_colors_original",
            "rep_topk_colors_full",
            "rep_topk_color_count_text_original",
            "rep_topk_color_count_text_full",
        ])

    case_df = row_df.groupby("case_key").agg(
        set_id=("set_id", "first"),
        target_item_id=("target_item_id", "first"),
        target_item_fg=("target_item_fg", "first"),
        query_color=("query_color", "first"),
        query_text_original=("query_text_original", "first"),
        query_text_full=("query_text_full", "first"),
        n_seeds=("seed", "size"),
        mean_rank_original=("rank_original", "mean"),
        mean_rank_full=("rank_full", "mean"),
        orig_issue_rate=("issue_original", "mean"),
        full_issue_rate=("issue_full", "mean"),
        orig_mean_alt_advantage=("alt_advantage_original", "mean"),
        full_mean_alt_advantage=("alt_advantage_full", "mean"),
        orig_target_match_rate=("target_match_original", "mean"),
        full_target_match_rate=("target_match_full", "mean"),
        orig_mean_query_count=("query_count_original", "mean"),
        full_mean_query_count=("query_count_full", "mean"),
        solved_seed_rate=("solved_row", "mean"),
        unsolved_seed_rate=("unsolved_row", "mean"),
        target_profile_text_original=("target_profile_text_original", "first"),
        target_profile_text_full=("target_profile_text_full", "first"),
        wrong_color_mode_original=("wrong_color_original", lambda s: safe_mode_non_none(s, default="none")),
        wrong_color_mode_full=("wrong_color_full", lambda s: safe_mode_non_none(s, default="none")),
    ).reset_index()

    case_df = case_df.merge(rep_df, on="case_key", how="left")

    if REQUIRE_TARGET_MATCH_QUERY:
        case_df = case_df[
            (case_df["orig_target_match_rate"] >= 0.5) &
            (case_df["full_target_match_rate"] >= 0.5)
        ].copy()

    # Overall solved / unsolved ratios
    orig_problem_cases = case_df[case_df["orig_issue_rate"] >= 0.5].copy()

    solved_cases = orig_problem_cases[orig_problem_cases["full_issue_rate"] < 0.5].copy()
    unsolved_cases = orig_problem_cases[orig_problem_cases["full_issue_rate"] >= 0.5].copy()

    total_problem_cases = len(orig_problem_cases)
    solved_ratio = len(solved_cases) / total_problem_cases if total_problem_cases else 0.0
    unsolved_ratio = len(unsolved_cases) / total_problem_cases if total_problem_cases else 0.0

    orig_problem_rows = row_df[row_df["issue_original"] == 1].copy()
    solved_rows = orig_problem_rows[orig_problem_rows["issue_full"] == 0].copy()
    unsolved_rows = orig_problem_rows[orig_problem_rows["issue_full"] == 1].copy()

    solved_row_ratio = len(solved_rows) / len(orig_problem_rows) if len(orig_problem_rows) else 0.0
    unsolved_row_ratio = len(unsolved_rows) / len(orig_problem_rows) if len(orig_problem_rows) else 0.0

    # Clear solved / clear unsolved for visualization
    clear_solved_cases = orig_problem_cases[
        (orig_problem_cases["orig_issue_rate"] >= CLEAR_ORIG_ISSUE_RATE_MIN) &
        (orig_problem_cases["full_issue_rate"] <= CLEAR_FULL_ISSUE_RATE_MAX) &
        (orig_problem_cases["orig_mean_alt_advantage"] >= CLEAR_ORIG_ALT_ADV_MIN) &
        (orig_problem_cases["full_mean_alt_advantage"] <= CLEAR_FULL_ALT_ADV_MAX) &
        (orig_problem_cases["full_mean_query_count"] >= orig_problem_cases["orig_mean_query_count"] + CLEAR_QUERY_COUNT_GAIN_MIN)
    ].copy()

    clear_unsolved_cases = orig_problem_cases[
        (orig_problem_cases["orig_issue_rate"] >= CLEAR_ORIG_ISSUE_RATE_MIN) &
        (orig_problem_cases["full_issue_rate"] >= CLEAR_FULL_ISSUE_RATE_MIN) &
        (orig_problem_cases["full_mean_alt_advantage"] >= CLEAR_FULL_ALT_ADV_MIN)
    ].copy()

    clear_middle_cases = orig_problem_cases[
        ~orig_problem_cases["case_key"].isin(clear_solved_cases["case_key"]) &
        ~orig_problem_cases["case_key"].isin(clear_unsolved_cases["case_key"])
    ].copy()

    clear_summary = pd.DataFrame([
        {
            "bucket": "clear_solved",
            "n_cases": len(clear_solved_cases),
            "ratio_over_orig_problem_cases": len(clear_solved_cases) / total_problem_cases if total_problem_cases else 0.0,
        },
        {
            "bucket": "clear_unsolved",
            "n_cases": len(clear_unsolved_cases),
            "ratio_over_orig_problem_cases": len(clear_unsolved_cases) / total_problem_cases if total_problem_cases else 0.0,
        },
        {
            "bucket": "middle_or_ambiguous",
            "n_cases": len(clear_middle_cases),
            "ratio_over_orig_problem_cases": len(clear_middle_cases) / total_problem_cases if total_problem_cases else 0.0,
        }
    ])

    print("\n" + "=" * 80)
    print("4) Overall ratio summary")
    print("=" * 80)
    print(f"Case-level denominator = cases with color-bias issue in original: {total_problem_cases}")
    print(f"solved_ratio   (case-level): {len(solved_cases)}/{total_problem_cases} = {solved_ratio:.4f}")
    print(f"unsolved_ratio (case-level): {len(unsolved_cases)}/{total_problem_cases} = {unsolved_ratio:.4f}")
    print()
    print(f"Row-level denominator = rows with color-bias issue in original: {len(orig_problem_rows)}")
    print(f"solved_ratio   (row-level): {len(solved_rows)}/{len(orig_problem_rows)} = {solved_row_ratio:.4f}")
    print(f"unsolved_ratio (row-level): {len(unsolved_rows)}/{len(orig_problem_rows)} = {unsolved_row_ratio:.4f}")

    overall_summary = pd.DataFrame([
        {
            "level": "case",
            "denominator": total_problem_cases,
            "solved_n": len(solved_cases),
            "solved_ratio": solved_ratio,
            "unsolved_n": len(unsolved_cases),
            "unsolved_ratio": unsolved_ratio,
        },
        {
            "level": "row",
            "denominator": len(orig_problem_rows),
            "solved_n": len(solved_rows),
            "solved_ratio": solved_row_ratio,
            "unsolved_n": len(unsolved_rows),
            "unsolved_ratio": unsolved_row_ratio,
        }
    ])
    display(overall_summary)

    print("\n" + "=" * 80)
    print("5) Clear-case summary for visualization")
    print("=" * 80)
    display(clear_summary)

    clear_solved_cases = clear_solved_cases.sort_values(
        ["orig_mean_alt_advantage", "full_mean_alt_advantage", "full_mean_query_count"],
        ascending=[False, True, False]
    ).reset_index(drop=True)

    clear_unsolved_cases = clear_unsolved_cases.sort_values(
        ["full_issue_rate", "full_mean_alt_advantage", "orig_mean_alt_advantage"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    print("[Clear solved cases]")
    display(clear_solved_cases[[
        "set_id", "target_item_id", "query_color",
        "orig_issue_rate", "full_issue_rate",
        "orig_mean_alt_advantage", "full_mean_alt_advantage",
        "orig_mean_query_count", "full_mean_query_count",
        "mean_rank_original", "mean_rank_full"
    ]].head(DISPLAY_TOPN))

    print("\n[Clear unsolved cases]")
    display(clear_unsolved_cases[[
        "set_id", "target_item_id", "query_color",
        "orig_issue_rate", "full_issue_rate",
        "orig_mean_alt_advantage", "full_mean_alt_advantage",
        "orig_mean_query_count", "full_mean_query_count",
        "mean_rank_original", "mean_rank_full"
    ]].head(DISPLAY_TOPN))

    if SAVE_SUMMARY_CSV:
        overall_summary.to_csv(os.path.join(OUTPUT_DIR, "overall_ratio_summary.csv"), index=False)
        clear_summary.to_csv(os.path.join(OUTPUT_DIR, "clear_case_summary.csv"), index=False)
        solved_cases.to_csv(os.path.join(OUTPUT_DIR, "solved_cases_all.csv"), index=False)
        unsolved_cases.to_csv(os.path.join(OUTPUT_DIR, "unsolved_cases_all.csv"), index=False)
        clear_solved_cases.to_csv(os.path.join(OUTPUT_DIR, "clear_solved_cases.csv"), index=False)
        clear_unsolved_cases.to_csv(os.path.join(OUTPUT_DIR, "clear_unsolved_cases.csv"), index=False)
        clear_middle_cases.to_csv(os.path.join(OUTPUT_DIR, "middle_or_ambiguous_cases.csv"), index=False)
        print(f"[saved] csv -> {OUTPUT_DIR}")

    print("\n" + "=" * 80)
    print("6) Load full outfits and visualize clear cases")
    print("=" * 80)
    outfit_map = load_outfit_map()

    print("\n--- CLEAR SOLVED EXAMPLES ---")
    for i in range(min(DISPLAY_TOPN, len(clear_solved_cases))):
        show_case(clear_solved_cases.iloc[i], case_type="CLEAR SOLVED", outfit_map=outfit_map)

    print("\n--- CLEAR UNSOLVED EXAMPLES ---")
    for i in range(min(DISPLAY_TOPN, len(clear_unsolved_cases))):
        show_case(clear_unsolved_cases.iloc[i], case_type="CLEAR UNSOLVED", outfit_map=outfit_map)

    print("\nDone.")


if __name__ == "__main__":
    main()

1) Load detail csv
detail_original: 17160
detail_full    : 17160

2) Load metadata
comparable rows: 745
need analyze images: 2880
  cached 500/2880
  cached 1000/2880
  cached 1500/2880
  cached 2000/2880
  cached 2500/2880

3) Analyze color-bias issue in original / full
NaN in alt_advantage_original: 0
NaN in alt_advantage_full    : 0

4) Overall ratio summary
Case-level denominator = cases with color-bias issue in original: 15
solved_ratio   (case-level): 5/15 = 0.3333
unsolved_ratio (case-level): 10/15 = 0.6667

Row-level denominator = rows with color-bias issue in original: 89
solved_ratio   (row-level): 39/89 = 0.4382
unsolved_ratio (row-level): 50/89 = 0.5618


,level,denominator,solved_n,solved_ratio,unsolved_n,unsolved_ratio
0,case,15,5,0.333333,10,0.666667
1,row,89,39,0.438202,50,0.561798



5) Clear-case summary for visualization


,bucket,n_cases,ratio_over_orig_problem_cases
0,clear_solved,3,0.2
1,clear_unsolved,9,0.6
2,middle_or_ambiguous,3,0.2


[Clear solved cases]


...


--- CLEAR UNSOLVED EXAMPLES ---


<Figure size 1650x1170 with 15 Axes>

<Figure size 1650x1170 with 15 Axes>

<Figure size 1650x1170 with 15 Axes>


Done.
